

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |




In [ ]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt python-dotenv email-validator pyngrok fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib transformers accelerate torch stopwordsiso deepface tf-keras opencv-python-headless mtcnn reportlab
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 10.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_sent_ud_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [ ]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE,
            role VARCHAR(20) NOT NULL DEFAULT 'employee')""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS role VARCHAR(20) NOT NULL DEFAULT 'employee'""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            mood_date DATE NOT NULL DEFAULT CURRENT_DATE,
            sentiment VARCHAR(20),
            emotion VARCHAR(30),
            compound_score REAL,
            confidence REAL,
            journal_text TEXT,
            source VARCHAR(10) NOT NULL DEFAULT 'manual',
            created_at TIMESTAMP NOT NULL DEFAULT NOW())""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS sentiment VARCHAR(20)""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS emotion VARCHAR(30)""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS compound_score REAL""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS journal_text TEXT""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS source VARCHAR(10) NOT NULL DEFAULT 'manual'""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS confidence REAL""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_mood_logs_user_date
            ON mood_logs(user_id, mood_date)""")


MOOD_LABELS = ["Happy", "Neutral", "Sad", "Stress", "Angry", "Fear"]

MOOD_EMOJI = {
    "Happy": "\U0001F60A",
    "Neutral": "\U0001F610",
    "Sad": "\U0001F622",
    "Stress": "\U0001F62B",
    "Angry": "\U0001F620",
    "Fear": "\U0001F628",
}


def save_manual_mood(user_id, mood_label):
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, source)
               VALUES (%s, %s, 'manual')""",
            (user_id, mood_label),
        )

def save_mood_log(user_id, sentiment, emotion, compound_score, journal_text, confidence=None):
    mood_label = emotion if emotion in MOOD_LABELS else "Neutral"
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, confidence, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, 'nlp')""",
            (user_id, mood_label, emotion, compound_score, confidence, journal_text),
        )

def get_mood_logs_for_month(user_id, year, month):
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (mood_date) mood_date, sentiment, emotion, compound_score, confidence, created_at
               FROM mood_logs
               WHERE user_id = %s
                 AND EXTRACT(YEAR FROM mood_date) = %s
                 AND EXTRACT(MONTH FROM mood_date) = %s
               ORDER BY mood_date, created_at DESC""",
            (user_id, year, month),
        )
        return cur.fetchall()

def get_user_mood_history(user_id, limit=200):
    with cursor() as cur:
        cur.execute(
            """SELECT mood_date, sentiment, emotion, compound_score, confidence, journal_text, source, created_at
               FROM mood_logs
               WHERE user_id = %s
               ORDER BY created_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_employee_mood_logs(limit_days=30):
    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.compound_score, m.confidence, m.created_at
               FROM mood_logs m
               JOIN users u ON u.id = m.user_id
               WHERE u.role = 'employee'
                 AND m.mood_date >= CURRENT_DATE - (%s || ' days')::interval
               ORDER BY m.mood_date DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()

def get_latest_mood_per_employee():
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (u.id) u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.confidence, m.created_at
               FROM users u
               JOIN mood_logs m ON m.user_id = u.id
               WHERE u.role = 'employee'
               ORDER BY u.id, m.created_at DESC"""
        )
        return cur.fetchall()


def save_face_scan(user_id, emotion, confidence):
    """Saves a face scan mood log"""
    mood_label = "Normal"
    if emotion in ["happy", "joy", "amazing"]: mood_label = "Happy"
    elif emotion in ["sad", "sadness"]: mood_label = "Sad"
    elif emotion in ["angry", "anger"]: mood_label = "Angry"

    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, confidence, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, 'face')""",
            (user_id, mood_label, emotion.capitalize(), 0.0, float(confidence), 'Face Scan', )
        )


Overwriting db.py


In [ ]:
from db import cursor

with cursor(commit=True) as cur:
    cur.execute("UPDATE mood_logs SET sentiment = 'Happy' WHERE sentiment = 'Amazing'")
    cur.execute("UPDATE mood_logs SET sentiment = 'Neutral' WHERE sentiment = 'Normal'")
print("Remapped legacy Amazing/Normal rows to Happy/Neutral.")

Remapped legacy Amazing/Normal rows to Happy/Neutral.


In [ ]:
%%writefile recommendations.py
"""
recommendations.py
Lightweight, dependency-free (no torch/spacy) home for the wellness
recommendation engine, shared by:
  - nlp_pipeline.py  -> get_recommendation() for a single journal entry
  - app.py           -> get_period_recommendation() for a Dashboard
                         date-range PDF export summary

Kept separate from nlp_pipeline.py so app.py (a plain Streamlit process)
doesn't need to import the heavy NLP stack just to build a report.
"""

# ---------------------------------------------------------------------------
# Wellness recommendation engine
#
# Simple rule-based recommender: maps a detected emotion label to a small set
# of curated wellness suggestions. This mirrors what a real MoodMentor-style
# system would do -- detected emotional state -> mapped intervention -- just
# without a database-backed content repository behind it.
#
# The confidence score (0-1) is used to pick *how* urgent/serious the
# suggestion should be for Sad/Stress/Angry/Fear:
#   - low confidence  (< 0.4): the model isn't very sure, so keep it light/generic
#   - medium confidence (0.4-0.7): a normal, matched coping suggestion
#   - high confidence (>= 0.7): the emotion signal is strong, so nudge more
#     firmly towards professional/structured support
#
# Happy and Neutral don't need an urgency ladder -- they always get an
# encouraging or maintenance-style tip instead.
# ---------------------------------------------------------------------------

WELLNESS_RECOMMENDATIONS = {
    "Happy": [
        "Great to see you're feeling good! Take a moment to note what contributed to this — it helps to recognize your own positive patterns.",
        "Keep this momentum going: consider sharing your positive energy with a colleague or teammate today.",
    ],
    "Neutral": [
        "A calm, steady mood is a good baseline. A short 5-minute walk or stretch break can help maintain it.",
        "Nothing urgent here — this could be a good time to plan your day or check in on a personal goal.",
    ],
    "Sad": {
        "low": "It looks like there might be a touch of sadness here. Consider writing a bit more in your journal about what's on your mind.",
        "medium": "Try a short guided breathing exercise (4 seconds in, 4 seconds hold, 4 seconds out) or step outside for a few minutes.",
        "high": "This seems like a strong low mood. Please consider talking to a trusted colleague, friend, or your HR/EAP wellness contact today.",
    },
    "Stress": {
        "low": "A little stress is normal — try a quick 2-minute breathing break before your next task.",
        "medium": "Consider breaking your current task into smaller steps, and take a 10-minute break away from your screen.",
        "high": "Your stress signal looks high. Try a longer break, deep breathing, or a short walk, and consider flagging your workload to your manager or HR.",
    },
    "Angry": {
        "low": "A bit of frustration is showing. A short pause before responding to anything stressful can help.",
        "medium": "Try stepping away for 5-10 minutes before continuing. Cognitive reframing — writing down the situation objectively — can help too.",
        "high": "This reads as strong frustration or anger. Please take a proper break away from the trigger, and consider talking it through with someone you trust or your HR/EAP contact.",
    },
    "Fear": {
        "low": "A little anxiety is showing. Grounding techniques (naming 5 things you can see, 4 you can hear) can help settle it.",
        "medium": "Try a short guided breathing or grounding exercise, and write down specifically what's worrying you — it often feels more manageable on paper.",
        "high": "This looks like a strong fear/anxiety signal. Please consider reaching out to a trusted colleague, your HR/EAP program, or a mental health professional.",
    },
}

# Maps the 5-point manual mood-picker label (db.MOOD_LABELS) onto the same
# 6-label emotion vocabulary above, so entries with no NLP/emotion data
# (manual mood taps) can still be folded into a recommendation.
MOOD_TO_EMOTION_BUCKET = {
    "Amazing": "Happy",
    "Happy": "Happy",
    "Normal": "Neutral",
    "Sad": "Sad",
    "Angry": "Angry",
}


def _confidence_bucket(confidence: float) -> str:
    """Buckets a 0-1 confidence score into low / medium / high urgency."""
    if confidence is None:
        return "medium"
    if confidence < 0.4:
        return "low"
    if confidence < 0.7:
        return "medium"
    return "high"


def get_recommendation(
    emotion_label: str,
    confidence: float = None,
    sentiment: str = None,
    sentiment_score: float = None,
) -> str:
    """
    Returns a wellness suggestion string, combining both classifiers:

    - `emotion_label` / `confidence` come from the BERT emotion model
      (Happy, Sad, Stress, Angry, Fear, Neutral).
    - `sentiment` / `sentiment_score` come from VADER (Positive, Negative,
      Neutral + a compound score from -1 to 1).

    These two models are trained independently and can disagree -- e.g. BERT
    says "Neutral" while VADER's compound score is clearly negative. Relying
    on the emotion label alone would then give a generic "maintenance" tip
    for text that actually reads negative.

    Fix: if BERT's top emotion is "Neutral" but VADER disagrees and calls the
    text "Negative", we treat it as mild Sad/Stress instead of Neutral, using
    the *sentiment* score for urgency instead of the (less reliable, in this
    case) emotion confidence. Otherwise, emotion label + emotion confidence
    drive the recommendation as before.
    """
    effective_label = emotion_label
    effective_confidence = confidence

    if emotion_label == "Neutral" and sentiment == "Negative":
        effective_label = "Sad"
        magnitude = abs(sentiment_score) if sentiment_score is not None else 0.3
        effective_confidence = magnitude  # -1..1 magnitude reused as 0..1 bucket input

    entry = WELLNESS_RECOMMENDATIONS.get(effective_label)
    if entry is None:
        return "Take a moment to check in with yourself today."

    if isinstance(entry, list):
        # Happy / Neutral (and genuinely neutral-sentiment text): no urgency
        # ladder, just rotate suggestions.
        import random
        return random.choice(entry)

    bucket = _confidence_bucket(effective_confidence)
    return entry[bucket]


def get_period_recommendation(entries: list[dict]) -> str:
    """
    Builds a short 2-3 sentence wellness summary for a *set* of mood_logs
    rows (e.g. everything within a Dashboard date-range export), rather
    than a single journal entry.

    Each `entries` item is expected to look like a row from
    db.get_user_mood_history(): at minimum `sentiment` (the 5-point mood
    label), and optionally `emotion` + `confidence` (present only for
    source == 'nlp' journal entries).

    Prefers the richer emotion/confidence data where available and falls
    back to the manual mood-picker label (mapped onto the same bucket
    vocabulary) otherwise, so a period made up of only emoji taps still
    gets a sensible recommendation.
    """
    if not entries:
        return "No entries were logged in this period yet."

    bucket_counts: dict[str, int] = {}
    bucket_confidences: dict[str, list[float]] = {}

    for e in entries:
        if e.get("source") == "nlp" and e.get("emotion"):
            bucket = e["emotion"]
            conf = e.get("confidence")
        else:
            bucket = MOOD_TO_EMOTION_BUCKET.get(e.get("sentiment"), "Neutral")
            conf = None

        bucket_counts[bucket] = bucket_counts.get(bucket, 0) + 1
        if conf is not None:
            bucket_confidences.setdefault(bucket, []).append(conf)

    total = sum(bucket_counts.values())
    dominant_bucket = max(bucket_counts, key=bucket_counts.get)
    dominant_count = bucket_counts[dominant_bucket]
    pct = round(100 * dominant_count / total)

    confs = bucket_confidences.get(dominant_bucket)
    avg_conf = sum(confs) / len(confs) if confs else None

    tip = get_recommendation(dominant_bucket, avg_conf)

    overview = (
        f"Over this period, {dominant_bucket.lower()} was your most common state "
        f"({dominant_count} of {total} entries, {pct}%)."
    )
    closing = "Keep logging regularly so trends like this are easier to catch early."

    return f"{overview} {tip} {closing}"


Overwriting recommendations.py


In [ ]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "role": user.get("role", "employee"),
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw, role="employee"):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash,role) VALUES (%s,%s,%s,%s)",
                    (username, email, hash_pw(pw), role))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True

Overwriting auth.py


In [ ]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)

Overwriting email_utils.py


In [ ]:
import os

folders = [
    "components",
    "styles",
    "assets",
    "assets/logo",
    "assets/backgrounds",
    "assets/illustrations",
    "assets/icons"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Done")

✅ Done


In [ ]:
%%writefile components/landing.py

# Landing page code will go here
# components/landing.py

"""
components/landing.py

Premium landing page for the Employee Wellness Management System.
Renders a pastel mountain hero background, centered logo, AI Powered
badges, hero title/subtitle, glassmorphism feature cards, and a
Get Started call-to-action that opens the auth panel.

Expects:
  - assets/backgrounds/background.png  (hero background image)
  - assets/logo/logo.png               (centered logo)
  - assets/css/landing.css             (stylesheet, loaded separately)
"""

import base64
import os
from pathlib import Path

import streamlit as st

# --------------------------------------------------------------------------- #
# Paths
# --------------------------------------------------------------------------- #
#
# Different launch environments (local venv, Colab, Docker, `streamlit run`
# from a different cwd, etc.) can make __file__-relative paths and the
# current working directory disagree about where the project root is.
# _resolve_path() checks several likely roots and returns the first path
# that actually exists, so assets load regardless of how the app is run.

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [
    _THIS_FILE.parent.parent,      # e.g. project_root/components/landing.py -> project_root
    Path.cwd(),                    # wherever `streamlit run` was launched from
    Path.cwd().parent,             # one level up from cwd, just in case
]


def _resolve_path(*relative_parts: str) -> Path:
    """Return the first existing path among candidate project roots.

    If none exist, returns the path under the primary (file-based) root so
    the caller still gets a sensible path to report in error messages.
    """
    for root in _CANDIDATE_ROOTS:
        candidate = root.joinpath(*relative_parts)
        if candidate.exists():
            return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*relative_parts)


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    """Like _resolve_path, but tries several different relative-path shapes.

    Useful when a file could plausibly live under more than one folder
    convention (e.g. `styles/landing.css` vs `assets/css/landing.css`).
    Each argument is a tuple of path parts. Returns the first that exists
    across all candidate roots, else the first shape under the primary root.
    """
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


def _get_paths() -> dict:
    """Resolve asset paths fresh on every call (cheap, avoids stale imports)."""
    return {
        "background": _resolve_path("assets", "backgrounds", "background.png"),
        "logo": _resolve_path("assets", "logo", "logo.png"),
        # CSS may live in a top-level `styles/` folder (this project's layout)
        # or the `assets/css/` convention -- try both.
        "css": _resolve_first(
            ("styles", "landing.css"),
            ("assets", "css", "landing.css"),
        ),
    }


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    """Read a local file and return its base64-encoded string."""
    try:
        with open(file_path, "rb") as f:
            data = f.read()
        return base64.b64encode(data).decode("utf-8")
    except FileNotFoundError:
        return ""


def _load_css(css_path: Path) -> bool:
    """Inject an external CSS file into the Streamlit app. Returns True if found."""
    if css_path.exists():
        with open(css_path, "r", encoding="utf-8") as f:
            css = f.read()
        st.markdown(f"<style>{css}</style>", unsafe_allow_html=True)
        return True
    return False


def _inject_background(image_path: Path) -> bool:
    """Set the page background image via base64 CSS. Returns True if found.

    Newer Streamlit versions render an inner `stAppViewContainer` div with
    its own opaque background-color that sits on top of `.stApp`, which
    silently hides a background-image set only on `.stApp`. We target every
    layer that could be painting over it and force those layers transparent.
    """
    encoded = _get_base64_of_file(str(image_path))
    if not encoded:
        return False
    st.markdown(
        f"""
        <style>
        .stApp,
        [data-testid="stAppViewContainer"],
        [data-testid="stMain"] {{
            background-color: transparent !important;
        }}

        [data-testid="stAppViewContainer"] {{
            background-image:
                linear-gradient(
                    180deg,
                    rgba(255, 255, 255, 0.12) 0%,
                    rgba(255, 255, 255, 0.05) 40%,
                    rgba(255, 255, 255, 0.18) 100%
                ),
                url("data:image/png;base64,{encoded}") !important;
            background-size: cover !important;
            background-position: center !important;
            background-repeat: no-repeat !important;
            background-attachment: fixed !important;
        }}

        [data-testid="stHeader"] {{
            background-color: transparent !important;
        }}
        </style>
        """,
        unsafe_allow_html=True,
    )
    return True


def _get_logo_html(logo_path: Path, max_width_px: int = 260) -> str:
    """Return an <img> tag for the logo, or a text fallback."""
    encoded = _get_base64_of_file(str(logo_path))
    if encoded:
        return (
            f'<img src="data:image/png;base64,{encoded}" '
            f'class="landing-logo-img" style="max-width:{max_width_px}px;" '
            f'alt="Company Logo" />'
        )
    return f'<div class="landing-logo-fallback">{BRAND_NAME_PRIMARY}{BRAND_NAME_ACCENT}</div>'


# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #

FEATURE_CARDS = [
    {
        "icon": "🙂",
        "title": "Understand",
        "tone": "indigo",
        "description": "Understand your emotions and gain clarity about your inner self.",
    },
    {
        "icon": "🌱",
        "title": "Improve",
        "tone": "green",
        "description": "Build healthy habits and improve your mental and emotional well-being.",
    },
    {
        "icon": "📈",
        "title": "Grow",
        "tone": "pink",
        "description": "Track your progress and grow into the best version of yourself.",
    },
]

AI_BADGES = [
    {"icon": "✨", "label": "AI Powered", "tone": "indigo"},
    {"icon": "🔒", "label": "Private &amp; Secure", "tone": "green"},
    {"icon": "💜", "label": "Always Here", "tone": "pink"},
]

BRAND_NAME_PRIMARY = "Mood"
BRAND_NAME_ACCENT = "Mentor"
BRAND_TAGLINE = "AI-Powered Emotional Wellness Assistant"
BRAND_DESCRIPTION = (
    "Understand your emotions, improve your well-being, and grow into "
    "the best version of yourself with the power of AI and self-care."
)


# --------------------------------------------------------------------------- #
# Section renderers
# --------------------------------------------------------------------------- #

def _render_badge_row() -> str:
    return "".join(
        f'<span class="ai-badge ai-badge--{b["tone"]}">'
        f'<span class="ai-badge-icon">{b["icon"]}</span>{b["label"]}</span>'
        for b in AI_BADGES
    )


def _render_hero(logo_html: str) -> None:
    st.markdown(
        f"""
        <div class="hero-container">
            <div class="landing-logo-wrapper">
                {logo_html}
            </div>
            <div class="ai-badge-row">
                {_render_badge_row()}
            </div>
            <p class="hero-eyebrow">Welcome to</p>
            <div class="hero-divider">
                <span class="hero-divider-line"></span>
                <span class="hero-divider-leaf">🌿</span>
                <span class="hero-divider-line"></span>
            </div>
            <h1 class="hero-title">
                {BRAND_NAME_PRIMARY}<span class="hero-title-accent">{BRAND_NAME_ACCENT}</span>
            </h1>
            <p class="hero-tagline">{BRAND_TAGLINE}</p>
            <p class="hero-subtitle">{BRAND_DESCRIPTION}</p>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_feature_cards() -> None:
    st.markdown('<div class="feature-grid">', unsafe_allow_html=True)

    cols = st.columns(len(FEATURE_CARDS), gap="medium")
    for col, feature in zip(cols, FEATURE_CARDS):
        with col:
            st.markdown(
                f"""
                <div class="glass-card">
                    <div class="glass-card-icon glass-card-icon--{feature['tone']}">
                        {feature['icon']}
                    </div>
                    <div class="glass-card-title glass-card-title--{feature['tone']}">
                        {feature['title']}
                    </div>
                    <div class="glass-card-desc">{feature['description']}</div>
                    <div class="glass-card-underline glass-card-underline--{feature['tone']}"></div>
                </div>
                """,
                unsafe_allow_html=True,
            )

    st.markdown("</div>", unsafe_allow_html=True)


def _render_cta() -> None:
    st.markdown('<div class="cta-wrapper">', unsafe_allow_html=True)

    left, center, right = st.columns([1.3, 1, 1.3])
    with center:
        clicked = st.button(
            "Get Started  →",
            key="landing_get_started_btn",
        )
        if clicked:
            st.session_state.show_auth_panel = True
            st.rerun()

    st.markdown("</div>", unsafe_allow_html=True)


def _render_footer() -> None:
    st.markdown(
        """
        <div class="landing-footer">
            <span class="privacy-pill">
                🛡️ Your privacy is our priority. Your journey is safe with us.
            </span>
        </div>
        """,
        unsafe_allow_html=True,
    )


# --------------------------------------------------------------------------- #
# Public entry point
# --------------------------------------------------------------------------- #

def render_landing_page(debug: bool = True) -> None:
    """Render the full premium landing page.

    Args:
        debug: When True (default), shows a small warning banner listing
            exact resolved paths for any missing asset (css/background/logo)
            so misplacement is easy to spot. Set to False once assets are
            confirmed in place.
    """

    if "show_auth_panel" not in st.session_state:
        st.session_state.show_auth_panel = False

    paths = _get_paths()

    css_ok = _load_css(paths["css"])
    bg_ok = _inject_background(paths["background"])

    if debug:
        missing = []
        if not css_ok:
            missing.append(f"CSS not found at: `{paths['css']}`")
        if not bg_ok:
            missing.append(f"Background image not found at: `{paths['background']}`")
        if not paths["logo"].exists():
            missing.append(f"Logo not found at: `{paths['logo']}`")
        if missing:
            st.warning(
                "Landing page assets missing (page will look unstyled until "
                "these exist at the exact paths below):\n\n"
                + "\n\n".join(f"- {m}" for m in missing)
                + f"\n\nSearched project roots (in order): "
                + ", ".join(f"`{r}`" for r in _CANDIDATE_ROOTS)
            )

    st.markdown('<div class="landing-page-container">', unsafe_allow_html=True)

    logo_html = _get_logo_html(paths["logo"])
    _render_hero(logo_html)
    _render_cta()
    st.markdown('<div class="section-spacer"></div>', unsafe_allow_html=True)
    _render_feature_cards()
    _render_footer()

    st.markdown("</div>", unsafe_allow_html=True)


# Alias for compatibility with callers that import `show_landing`
# (e.g. `from components.landing import show_landing`).
show_landing = render_landing_page


# Allow `components/landing.py` to be run directly for isolated preview.
if __name__ == "__main__":
    st.set_page_config(
        page_title="Employee Wellness Management System",
        page_icon="🌿",
        layout="wide",
        initial_sidebar_state="collapsed",
    )
    render_landing_page()

Overwriting components/landing.py


In [ ]:
%%writefile styles/landing.css

/* ===========================================================
   Employee Wellness Management System
   Premium Landing Page
   landing.css
   =========================================================== */


/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* ==========================================================================
   assets/css/landing.css
   Premium pastel-mountain landing page styles for the
   Employee Wellness Management System.
   Loaded by components/landing.py via _load_css().
   ========================================================================== */

/* -------------------------------------------------------------------------
   Root tokens
   ------------------------------------------------------------------------- */
:root {
  --wellness-primary: #7c6fe8;
  --wellness-primary-dark: #5b4fc4;
  --wellness-accent: #a78bfa;
  --wellness-accent-2: #c4b5fd;
  --wellness-ink: #241f3d;
  --wellness-muted: #6b6584;

  --tone-indigo: #6d5ce8;
  --tone-indigo-bg: rgba(124, 111, 232, 0.14);
  --tone-green: #2f9e6e;
  --tone-green-bg: rgba(47, 158, 110, 0.14);
  --tone-pink: #e0568c;
  --tone-pink-bg: rgba(224, 86, 140, 0.14);

  --glass-bg: rgba(255, 255, 255, 0.45);
  --glass-bg-strong: rgba(255, 255, 255, 0.68);
  --glass-border: rgba(255, 255, 255, 0.7);
  --glass-shadow: 0 8px 32px rgba(70, 50, 120, 0.15);
  --radius-lg: 24px;
  --radius-md: 16px;
  --radius-pill: 999px;
}

/* -------------------------------------------------------------------------
   Global cleanup / Streamlit chrome
   ------------------------------------------------------------------------- */
html, body, [class^="css"] {
  font-family: "Poppins", "Segoe UI", -apple-system, BlinkMacSystemFont,
    sans-serif;
}

#MainMenu,
footer,
header[data-testid="stHeader"] {
  background: transparent;
}

section[data-testid="stSidebar"] {
  display: none !important;
  width: 0px !important;
  min-width: 0px !important;
  max-width: 0px !important;
}

button[data-testid="stSidebarCollapsedControl"],
div[data-testid="stSidebarCollapsedControl"],
[data-testid="collapsedControl"] {
  display: none !important;
}

div[data-testid="stAppViewContainer"] > .main,
section.main {
  margin-left: 0 !important;
  width: 100% !important;
  max-width: 100% !important;
}

.block-container {
  padding-top: 3.5rem !important;
  padding-bottom: 3rem !important;
  max-width: 1200px !important;
  margin-left: auto !important;
  margin-right: auto !important;
}

/* -------------------------------------------------------------------------
   Page container
   ------------------------------------------------------------------------- */
.landing-page-container {
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  text-align: center;
  width: 100%;
  margin: 0 auto;
  animation: landing-fade-in 0.9s ease-out;
}

.section-spacer {
  height: 3rem;
  width: 100%;
}

@keyframes landing-fade-in {
  from {
    opacity: 0;
    transform: translateY(16px);
  }
  to {
    opacity: 1;
    transform: translateY(0);
  }
}

/* -------------------------------------------------------------------------
   Hero section
   ------------------------------------------------------------------------- */
.hero-container {
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  gap: 0.9rem;
  max-width: 760px;
  margin: 0 auto;
  padding: 1.5rem 1.5rem 1rem;
  width: 100%;
}

.landing-logo-wrapper {
  display: flex;
  align-items: center;
  justify-content: center;
  padding: 0;
  margin-bottom: 0.25rem;
  animation: landing-float 5s ease-in-out infinite;
}

.landing-logo-img {
  display: block;
  height: auto;
  width: 100%;
  object-fit: contain;
  filter: drop-shadow(0 6px 18px rgba(70, 50, 120, 0.18));
}

.landing-logo-fallback {
  font-size: 1.6rem;
  font-weight: 800;
  color: var(--wellness-ink);
  letter-spacing: -0.02em;
}

@keyframes landing-float {
  0%, 100% { transform: translateY(0px); }
  50% { transform: translateY(-8px); }
}

.hero-eyebrow {
  margin: 0.75rem 0 0;
  font-size: 1.15rem;
  font-weight: 600;
  color: var(--wellness-ink);
  text-shadow: 0 2px 12px rgba(255, 255, 255, 0.6);
}

.hero-divider {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 0.5rem;
  margin: 0.15rem 0 0.35rem;
}

.hero-divider-line {
  width: 48px;
  height: 1px;
  background: linear-gradient(
    90deg,
    transparent 0%,
    var(--wellness-accent) 100%
  );
}

.hero-divider-line:last-child {
  background: linear-gradient(
    90deg,
    var(--wellness-accent) 0%,
    transparent 100%
  );
}

.hero-divider-leaf {
  font-size: 0.9rem;
  opacity: 0.8;
}

.hero-title {
  font-size: clamp(2.6rem, 6vw, 4.4rem);
  font-weight: 800;
  color: var(--wellness-ink);
  line-height: 1.05;
  margin: 0;
  letter-spacing: -0.03em;
  text-shadow: 0 4px 24px rgba(255, 255, 255, 0.5);
}

.hero-title-accent {
  background: linear-gradient(
    100deg,
    var(--wellness-primary-dark) 0%,
    var(--wellness-primary) 55%,
    var(--wellness-accent) 100%
  );
  -webkit-background-clip: text;
  background-clip: text;
  -webkit-text-fill-color: transparent;
}

.hero-tagline {
  margin: 0.5rem 0 0;
  font-size: clamp(1.05rem, 1.6vw, 1.3rem);
  font-weight: 700;
  color: var(--wellness-primary-dark);
  text-shadow: 0 2px 12px rgba(255, 255, 255, 0.6);
}

.hero-subtitle {
  max-width: 620px;
  font-size: clamp(1rem, 1.4vw, 1.15rem);
  color: var(--wellness-ink);
  line-height: 1.65;
  margin: 0.6rem auto 0;
  text-shadow: 0 2px 14px rgba(255, 255, 255, 0.55);
}

/* -------------------------------------------------------------------------
   AI Powered badge row
   ------------------------------------------------------------------------- */
.ai-badge-row {
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  justify-content: center;
  gap: 0.6rem;
  margin: 0.25rem 0;
}

.ai-badge {
  display: inline-flex;
  align-items: center;
  gap: 0.4rem;
  padding: 0.55rem 1.15rem;
  font-size: 0.85rem;
  font-weight: 600;
  color: var(--wellness-primary-dark);
  background: rgba(255, 255, 255, 0.85);
  border: 1px solid rgba(255, 255, 255, 0.95);
  border-radius: var(--radius-pill);
  box-shadow: 0 6px 18px rgba(70, 50, 120, 0.12);
  backdrop-filter: blur(6px);
  -webkit-backdrop-filter: blur(6px);
  letter-spacing: 0.01em;
  transition: transform 0.25s ease, box-shadow 0.25s ease;
}

.ai-badge-icon {
  font-size: 0.95rem;
  line-height: 1;
}

.ai-badge--indigo { color: var(--tone-indigo); }
.ai-badge--green { color: var(--tone-green); }
.ai-badge--pink { color: var(--tone-pink); }

.ai-badge:hover {
  transform: translateY(-2px);
  box-shadow: 0 6px 18px rgba(70, 50, 120, 0.14);
}

/* -------------------------------------------------------------------------
   Feature cards (glassmorphism)
   ------------------------------------------------------------------------- */
.feature-grid {
  width: 100%;
  max-width: 980px;
  margin: 0.5rem auto 0;
}

.glass-card {
  height: 100%;
  min-height: 210px;
  display: flex;
  flex-direction: column;
  align-items: center;
  text-align: center;
  gap: 0.6rem;
  padding: 1.8rem 1.4rem;
  background: rgba(255, 255, 255, 0.88);
  border: 1px solid rgba(255, 255, 255, 0.9);
  border-radius: var(--radius-lg);
  box-shadow: 0 12px 34px rgba(70, 50, 120, 0.16);
  backdrop-filter: blur(6px);
  -webkit-backdrop-filter: blur(6px);
  transition: transform 0.3s ease, box-shadow 0.3s ease;
}

.glass-card:hover {
  transform: translateY(-6px);
  box-shadow: 0 16px 42px rgba(70, 50, 120, 0.24);
}

.glass-card-icon {
  font-size: 2.1rem;
  width: 64px;
  height: 64px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: rgba(255, 255, 255, 0.55);
  box-shadow: inset 0 0 0 1px rgba(255, 255, 255, 0.7);
}

.glass-card-icon--indigo { background: var(--tone-indigo-bg); }
.glass-card-icon--green { background: var(--tone-green-bg); }
.glass-card-icon--pink { background: var(--tone-pink-bg); }

.glass-card-title {
  font-size: 1.1rem;
  font-weight: 700;
  color: var(--wellness-ink);
}

.glass-card-title--indigo { color: var(--tone-indigo); }
.glass-card-title--green { color: var(--tone-green); }
.glass-card-title--pink { color: var(--tone-pink); }

.glass-card-desc {
  font-size: 0.88rem;
  color: var(--wellness-muted);
  line-height: 1.55;
}

.glass-card-underline {
  width: 36px;
  height: 3px;
  border-radius: var(--radius-pill);
  margin-top: 0.25rem;
}

.glass-card-underline--indigo { background: var(--tone-indigo); }
.glass-card-underline--green { background: var(--tone-green); }
.glass-card-underline--pink { background: var(--tone-pink); }

/* -------------------------------------------------------------------------
   Call to action / Get Started button
   ------------------------------------------------------------------------- */
.cta-wrapper {
  width: 100%;
  margin: 0.5rem 0 1rem;
  display: flex;
  justify-content: center;
}

/* NOTE: separate st.markdown() calls never actually nest around widgets
   rendered in between them -- each is an isolated block in Streamlit's
   DOM, so `.cta-wrapper [data-testid="stButton"]` never matches anything.
   We target Streamlit's button globally instead (safe here since this is
   the only st.button on the page). */
div[data-testid="stButton"] {
  display: flex;
  justify-content: center;
}

div[data-testid="stButton"] button,
div[data-testid="stButton"] button[kind="secondary"],
div[data-testid="stButton"] button[kind="primary"],
div[data-testid="baseButton-secondary"],
div[data-testid="baseButton-primary"],
div[data-testid="stBaseButton-secondary"],
div[data-testid="stBaseButton-primary"] {
  width: auto !important;
  min-width: 240px !important;
  padding: 1rem 2.6rem !important;
  font-size: 1.15rem !important;
  font-weight: 700 !important;
  color: #ffffff !important;
  background: linear-gradient(
    100deg,
    var(--wellness-primary-dark) 0%,
    var(--wellness-primary) 100%
  ) !important;
  border: none !important;
  border-radius: var(--radius-pill) !important;
  box-shadow: 0 12px 32px rgba(91, 79, 196, 0.45) !important;
  transition: transform 0.25s ease, box-shadow 0.25s ease, filter 0.25s ease !important;
}

div[data-testid="stButton"] button p,
div[data-testid="stButton"] button div,
div[data-testid="stButton"] button span {
  color: #ffffff !important;
  font-size: 1.15rem !important;
  font-weight: 700 !important;
  margin: 0 !important;
}

div[data-testid="stButton"] button:hover {
  transform: translateY(-3px) scale(1.02);
  box-shadow: 0 16px 40px rgba(91, 79, 196, 0.55) !important;
  filter: brightness(1.06);
}

div[data-testid="stButton"] button:active {
  transform: translateY(0px) scale(0.99);
}

div[data-testid="stButton"] button:focus {
  outline: none !important;
  box-shadow: 0 0 0 4px rgba(124, 111, 232, 0.35) !important;
}

/* -------------------------------------------------------------------------
   Footer
   ------------------------------------------------------------------------- */
.landing-footer {
  margin-top: 2.5rem;
  padding-top: 0.5rem;
  display: flex;
  justify-content: center;
}

.privacy-pill {
  display: inline-flex;
  align-items: center;
  gap: 0.5rem;
  padding: 0.65rem 1.4rem;
  font-size: 0.85rem;
  font-weight: 600;
  color: var(--wellness-primary-dark);
  background: var(--glass-bg-strong);
  border: 1px solid var(--glass-border);
  border-radius: var(--radius-pill);
  box-shadow: var(--glass-shadow);
  backdrop-filter: blur(10px);
  -webkit-backdrop-filter: blur(10px);
}

/* -------------------------------------------------------------------------
   Responsive tweaks
   ------------------------------------------------------------------------- */
@media (max-width: 768px) {
  .landing-logo-wrapper {
    width: 96px;
    height: 96px;
  }

  .hero-container {
    padding: 1.5rem 1rem 1rem;
  }

  .glass-card {
    min-height: 180px;
    padding: 1.4rem 1.1rem;
  }
}

Overwriting styles/landing.css


In [ ]:
%%writefile components/auth.py
"""
components/auth.py

Premium authentication UI for MoodMentor, matching the landing page's
pastel-mountain glassmorphism design language.

IMPORTANT: This module is UI ONLY.
Every backend call (auth.*, email_utils.send_otp), every st.session_state
key (auth_mode, email, token, show_auth_panel, page), and every branch of
the login / signup / verify / forgot / reset flow is copied verbatim from
the original inline block in app.py. No authentication, JWT, OTP, or
database logic has been changed -- only the markup/CSS around it.

Deliberately NOT included (no backend support exists for these yet):
  - "Continue with Google" / "Continue with Microsoft" (no OAuth backend)
  - "Remember Me" checkbox (nothing to wire it to)

Expects:
  - assets/backgrounds/background.png  (shared with the landing page)
  - styles/auth.css                    (primary stylesheet location;
    also checked: assets/styles/auth.css, assets/css/auth.css)

Usage in app.py:

    from components.auth import render_auth_screen
    ...
    if st.session_state.page == "welcome":
        if not st.session_state.show_auth_panel:
            show_landing()
            st.stop()
        render_auth_screen()
        st.stop()
"""

import base64
import re
from pathlib import Path

import streamlit as st

from auth import (
    make_token, get_user, username_taken, create_user, verify_user,
    set_password, check_pw, new_otp, save_otp, check_otp,
)
from email_utils import send_otp


# --------------------------------------------------------------------------- #
# Paths (same robust multi-root resolver pattern as components/landing.py)
# --------------------------------------------------------------------------- #

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [
    _THIS_FILE.parent.parent,      # project_root/components/auth.py -> project_root
    Path.cwd(),
    Path.cwd().parent,
]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    """Return the first existing path across candidate roots and shapes."""
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


def _get_paths() -> dict:
    return {
        "background": _resolve_first(("assets", "backgrounds", "background.png")),
        "logo": _resolve_first(("assets", "logo", "logo.png")),
        # `styles/auth.css` matches this project's actual layout; the other
        # two are fallbacks in case the project is reorganized later.
        "css": _resolve_first(
            ("styles", "auth.css"),
            ("assets", "styles", "auth.css"),
            ("assets", "css", "auth.css"),
        ),
    }


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _load_css(css_path: Path) -> bool:
    if css_path.exists():
        st.markdown(
            f"<style>{css_path.read_text(encoding='utf-8')}</style>",
            unsafe_allow_html=True,
        )
        return True
    return False


def _inject_background(image_path: Path) -> bool:
    """Same technique used on the landing page: target stAppViewContainer
    directly and force the layers above it transparent, since that inner
    container paints an opaque background over a plain `.stApp` rule."""
    encoded = _get_base64_of_file(str(image_path))
    if not encoded:
        return False
    st.markdown(
        f"""
        <style>
        .stApp,
        [data-testid="stAppViewContainer"],
        [data-testid="stMain"] {{
            background-color: transparent !important;
        }}
        [data-testid="stAppViewContainer"] {{
            background-image:
                linear-gradient(
                    180deg,
                    rgba(255, 255, 255, 0.12) 0%,
                    rgba(255, 255, 255, 0.05) 40%,
                    rgba(255, 255, 255, 0.18) 100%
                ),
                url("data:image/png;base64,{encoded}") !important;
            background-size: cover !important;
            background-position: center !important;
            background-repeat: no-repeat !important;
            background-attachment: fixed !important;
        }}
        [data-testid="stHeader"] {{
            background-color: transparent !important;
        }}
        </style>
        """,
        unsafe_allow_html=True,
    )
    return True


# --------------------------------------------------------------------------- #
# UI-only helpers (copied from app.py so this module has no import-time
# dependency on app.py, which would create a circular import)
# --------------------------------------------------------------------------- #

def _valid_pw(pw: str) -> bool:
    return bool(len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw))


def _goto_auth(mode: str) -> None:
    st.session_state.auth_mode = mode
    st.rerun()


# --------------------------------------------------------------------------- #
# Left panel content (marketing / feature panel)
# --------------------------------------------------------------------------- #

FEATURES = [
    {
        "icon": "🧠",
        "title": "AI Mood Analysis",
        "desc": "Detect emotions from journal entries using NLP.",
    },
    {
        "icon": "📊",
        "title": "Wellness Analytics",
        "desc": "Track mood trends and streaks with visual dashboards.",
    },
    {
        "icon": "💬",
        "title": "MoodMentor Assistant",
        "desc": "Your 24/7 AI companion for supportive conversations.",
    },
    {
        "icon": "📸",
        "title": "Face Detection",
        "desc": "DeepFace-powered scans with personalized recommendations.",
    },
    {
        "icon": "🛡️",
        "title": "Secure & Private",
        "desc": "JWT-secured accounts and encrypted sessions.",
    },
]

def _get_logo_html(logo_path: Path, max_width_px: int = 220) -> str:
    encoded = _get_base64_of_file(str(logo_path))
    if encoded:
        return (
            f'<img src="data:image/png;base64,{encoded}" '
            f'class="auth-header-logo-img" style="max-width:{max_width_px}px;" '
            f'alt="Logo" />'
        )
    return '<div class="auth-header-logo-fallback">MoodMentor</div>'


def _render_top_header(logo_html: str) -> None:
    st.markdown(
        f"""
        <div class="auth-header">
            <div class="auth-header-logo">{logo_html}</div>
            <h1 class="auth-header-title">Employee Wellness Management System</h1>
            <p class="auth-header-subtitle">
                <span>➜</span> AI-Powered Workplace Wellbeing Platform <span>⬅</span>
            </p>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_floating_cards() -> None:
    st.markdown(
        """
        <div class="auth-float-card auth-float-card--left">
            <div class="auth-float-icon">💜</div>
            <div>
                <div class="auth-float-label">Mood Score</div>
                <div class="auth-float-value">84%</div>
                <div class="auth-float-tag">● Excellent</div>
            </div>
        </div>
        <div class="auth-float-card auth-float-card--right">
            <div class="auth-float-icon">🙂</div>
            <div>
                <div class="auth-float-label">Today's Wellness</div>
                <div class="auth-float-value">Excellent</div>
                <div class="auth-float-tag">Keep it up! 🌱</div>
            </div>
        </div>
        <div class="auth-float-card auth-float-card--bottom-right">
            <div class="auth-float-icon">🤖</div>
            <div>
                <div class="auth-float-label">AI Analysis</div>
                <div class="auth-float-value">Ready</div>
                <div class="auth-float-tag">Tap to view →</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_mockup_preview() -> None:
    """Purely decorative dashboard preview (donut + trend chart + chat
    bubble) -- illustrative only, not wired to real data. Mirrors the
    reference screenshot's right-hand mini preview inside the feature panel."""
    st.markdown(
        """
        <div class="auth-mockup">
            <div class="auth-mockup-card">
                <div class="auth-mockup-title">Wellness Overview</div>
                <div class="auth-mockup-donut-row">
                    <div class="auth-mockup-donut">
                        <div class="auth-mockup-donut-inner">
                            <span class="auth-mockup-donut-value">84</span>
                            <span class="auth-mockup-donut-label">Excellent</span>
                        </div>
                    </div>
                    <svg class="auth-mockup-spark" viewBox="0 0 120 40" preserveAspectRatio="none">
                        <polyline points="0,28 15,20 30,24 45,10 60,16 75,6 90,14 105,4 120,10"
                                  fill="none" stroke="#7c6fe8" stroke-width="2.5"
                                  stroke-linecap="round" stroke-linejoin="round"/>
                    </svg>
                </div>
                <div class="auth-mockup-axis">Mon Tue Wed Thu Fri Sat Sun</div>
            </div>
            <div class="auth-mockup-card">
                <div class="auth-mockup-title">Mood Trend (This Week)</div>
                <svg class="auth-mockup-spark auth-mockup-spark--wide" viewBox="0 0 220 60" preserveAspectRatio="none">
                    <polyline points="0,42 30,30 60,38 90,15 120,25 150,8 180,20 220,12"
                              fill="none" stroke="#a78bfa" stroke-width="3"
                              stroke-linecap="round" stroke-linejoin="round"/>
                </svg>
                <div class="auth-mockup-axis">Mon Tue Wed Thu Fri Sat Sun</div>
            </div>
            <div class="auth-mockup-chat">
                <div class="auth-mockup-bubble">
                    👋 Hi there!<br>I'm MoodMentor.<br>How can I help you today?
                </div>
                <div class="auth-mockup-avatar">🤖</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_left_panel() -> None:
    # Marker div lets auth.css target this specific glass card via :has()
    # without relying on separate st.markdown calls actually nesting
    # (they don't -- see note in auth.css).
    st.markdown('<div class="auth-left-marker"></div>', unsafe_allow_html=True)

    features_html = "".join(
        f'<div class="auth-feature-item">'
        f'<div class="auth-feature-icon">{f["icon"]}</div>'
        f'<div><div class="auth-feature-title">{f["title"]}</div>'
        f'<div class="auth-feature-desc">{f["desc"]}</div></div>'
        f"</div>"
        for f in FEATURES
    )

    text_col, preview_col = st.columns([1.15, 1], gap="medium")

    with text_col:
        st.markdown(
            f"""
            <div class="auth-left-content">
                <span class="auth-left-badge">AI &bull; Insights &bull; Wellbeing</span>
                <h2 class="auth-left-title">Empower Your Team.<br>Elevate Wellbeing.</h2>
                <p class="auth-left-desc">
                    Understand emotions, improve wellbeing, and build a
                    happier, healthier workplace with the power of AI.
                </p>
                <div class="auth-feature-list">{features_html}</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    with preview_col:
        _render_mockup_preview()


# --------------------------------------------------------------------------- #
# Right panel: the actual auth forms
# (identical logic to the original inline block -- only markup/labels
# gained icons; every condition, session_state key, and function call is
# unchanged)
# --------------------------------------------------------------------------- #

def _render_right_panel() -> None:
    st.markdown('<div class="auth-right-marker"></div>', unsafe_allow_html=True)

    mode = st.session_state.auth_mode

    if mode == "login":
        st.markdown('<div class="auth-heading">👋 Welcome Back!</div>', unsafe_allow_html=True)
        st.markdown(
            '<p class="auth-subtitle">Sign in to continue your wellness journey.</p>',
            unsafe_allow_html=True,
        )
        with st.form("login"):
            email = st.text_input("📧 Email Address", placeholder="Enter your email")
            pw = st.text_input("🔒 Password", type="password", placeholder="Enter your password")
            go = st.form_submit_button("Continue  ", type="primary", use_container_width=True)
        if go:
            u = get_user(email.strip().lower())
            if not u or not check_pw(pw, u["password_hash"]):
                st.error("Invalid email or password.")
            elif not u["is_verified"]:
                st.warning("Verify your email first.")
                st.session_state.email = u["email"]
                _goto_auth("verify")
            else:
                st.session_state.token = make_token(u)
                st.rerun()

        # Forgot password link, right-aligned
        st.markdown('<div class="auth-forgot-link">', unsafe_allow_html=True)
        if st.button("Forgot Password?", key="login_forgot_btn"):
            _goto_auth("forgot")
        st.markdown("</div>", unsafe_allow_html=True)

        st.markdown('<div class="auth-divider"><span>OR</span></div>', unsafe_allow_html=True)

        st.markdown('<div class="auth-create-account">', unsafe_allow_html=True)
        if st.button("Don't have an account?  Create Account →", key="login_create_account_btn", use_container_width=True):
            _goto_auth("signup")
        st.markdown("</div>", unsafe_allow_html=True)

    elif mode == "signup":
        st.markdown('<div class="auth-heading">✨ Create Account</div>', unsafe_allow_html=True)
        st.markdown('<p class="auth-subtitle">Let\'s get you started.</p>', unsafe_allow_html=True)
        with st.form("signup"):
            username = st.text_input("👤 Full Name", placeholder="Enter your full name")
            email = st.text_input("📧 Email Address", placeholder="Enter your email")
            pw = st.text_input("🔒 Password", type="password", placeholder="Create password")
            role_label = st.radio("I am signing up as a:", ["Employee", "Manager"], horizontal=True)
            go = st.form_submit_button("Send OTP  🚀", type="primary", use_container_width=True)
        if go:
            email = email.strip().lower()
            role = "manager" if role_label == "Manager" else "employee"
            if len(username) < 3:
                st.error("Username too short.")
            elif not _valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif username_taken(username) or get_user(email):
                st.error("Username or email already in use.")
            else:
                create_user(username, email, pw, role=role)
                code = new_otp()
                save_otp(email, code, "signup")
                ok, msg = send_otp(email, code, "signup")
                if ok:
                    st.session_state.email = email
                    st.success("Check your email for the code.")
                    _goto_auth("verify")
                else:
                    st.error(f"Email failed: {msg}")

        if st.button("Already have an account? Login", use_container_width=True):
            _goto_auth("login")

    elif mode == "verify":
        email = st.session_state.email
        st.markdown('<div class="auth-heading">📩 Verify OTP</div>', unsafe_allow_html=True)
        st.markdown(
            f'<p class="auth-subtitle">We sent a 6-digit code to <b>{email}</b></p>',
            unsafe_allow_html=True,
        )
        with st.form("verify"):
            code = st.text_input("🔢 Code", max_chars=6, placeholder="Enter 6-digit code")
            go = st.form_submit_button("Verify OTP  🚀", type="primary", use_container_width=True)
        if go:
            if check_otp(email, code.strip(), "signup"):
                verify_user(email)
                st.success("Verified! Please log in.")
                _goto_auth("login")
            else:
                st.error("Invalid or expired code.")

        if st.button("← Back to login", use_container_width=True):
            _goto_auth("login")

    elif mode == "forgot":
        st.markdown('<div class="auth-heading">🔑 Forgot Password</div>', unsafe_allow_html=True)
        st.markdown(
            '<p class="auth-subtitle">We\'ll send a reset code to your email.</p>',
            unsafe_allow_html=True,
        )
        with st.form("forgot"):
            email = st.text_input("📧 Your Account Email")
            go = st.form_submit_button("Send Reset Code  🚀", type="primary", use_container_width=True)
        if go:
            email = email.strip().lower()
            if get_user(email):
                code = new_otp()
                save_otp(email, code, "password_reset")
                send_otp(email, code, "password_reset")
            st.session_state.email = email
            st.info("If that email exists, a code was sent.")
            _goto_auth("reset")

        if st.button("← Back to login", use_container_width=True):
            _goto_auth("login")

    elif mode == "reset":
        email = st.session_state.email
        st.markdown('<div class="auth-heading">🔐 Reset Password</div>', unsafe_allow_html=True)
        st.markdown('<p class="auth-subtitle">Choose a new password.</p>', unsafe_allow_html=True)
        with st.form("reset"):
            code = st.text_input("🔢 Reset Code", max_chars=6)
            pw = st.text_input("🔒 New Password", type="password")
            go = st.form_submit_button("Reset Password  🚀", type="primary", use_container_width=True)
        if go:
            if not _valid_pw(pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif not check_otp(email, code.strip(), "password_reset"):
                st.error("Invalid or expired code.")
            else:
                set_password(email, pw)
                st.success("Password reset. Please log in.")
                _goto_auth("login")

        if st.button("← Back to login", use_container_width=True):
            _goto_auth("login")


# --------------------------------------------------------------------------- #
# Public entry point
# --------------------------------------------------------------------------- #

def _render_footer_bar() -> None:
    st.markdown(
        """
        <div class="auth-footer-bar">
            <span>🛡️ Your privacy is our priority. Your journey is safe with us.</span>
            <span class="auth-footer-sep">|</span>
            <span>✨ AI-Powered</span>
            <span class="auth-footer-sep">|</span>
            <span>🔒 Secure</span>
            <span class="auth-footer-sep">|</span>
            <span>👁️ Private</span>
            <span class="auth-footer-sep">|</span>
            <span class="auth-footer-highlight">👥 Trusted by 500+ Organizations</span>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_auth_screen() -> None:
    """Render the full premium auth screen (login/signup/verify/forgot/reset).

    Drop-in replacement for the inline `left, right = st.columns(...)` block
    that used to live in app.py. Reads/writes the exact same
    st.session_state keys (auth_mode, email, token) so nothing about the
    control flow changes -- only the visual layer.
    """
    paths = _get_paths()
    _load_css(paths["css"])
    _inject_background(paths["background"])

    logo_html = _get_logo_html(paths["logo"])
    _render_top_header(logo_html)
    _render_floating_cards()

    left, right = st.columns([1.35, 1], gap="large")

    with left:
        with st.container(border=True):
            _render_left_panel()

    with right:
        with st.container(border=True):
            _render_right_panel()

    _render_footer_bar()

Overwriting components/auth.py


In [ ]:
%%writefile styles/auth.css
/* ==========================================================================
   styles/auth.css
   Premium pastel-mountain auth screen styles for MoodMentor.
   Loaded by components/auth.py via _load_css(). Only applies while the
   auth screen is being rendered (page == "welcome" and show_auth_panel).

   NOTE ON NESTING: st.markdown('<div>') ... st.markdown('</div>') does
   NOT actually wrap Streamlit widgets rendered in between -- each
   st.markdown() call is its own isolated element. We rely on
   st.container(border=True) for real glass-card wrapping instead, and
   use a small marker div as the first child of each container so CSS
   can tell the two apart via :has().
   ========================================================================== */

:root {
  --auth-primary: #7c6fe8;
  --auth-primary-dark: #5b4fc4;
  --auth-accent: #a78bfa;
  --auth-ink: #241f3d;
  --auth-muted: #6b6584;
  --auth-glass: rgba(255, 255, 255, 0.86);
  --auth-glass-strong: rgba(255, 255, 255, 0.94);
  --auth-border: rgba(255, 255, 255, 0.95);
  --auth-radius: 28px;
  --auth-shadow: 0 24px 64px rgba(70, 50, 120, 0.22);
}

/* -------------------------------------------------------------------------
   Page chrome
   ------------------------------------------------------------------------- */
.block-container {
  padding-top: 5.5rem !important;
  padding-bottom: 3rem !important;
  max-width: 1240px !important;
}

/* -------------------------------------------------------------------------
   Top header: logo + title + subtitle
   ------------------------------------------------------------------------- */
.auth-header {
  display: flex;
  flex-direction: column;
  align-items: center;
  text-align: center;
  gap: 0.4rem;
  padding: 0 1rem 1.75rem;
  animation: auth-panel-in 0.5s ease-out both;
}

.auth-header-logo {
  margin-bottom: 0.4rem;
}

.auth-header-logo-img {
  display: block;
  height: auto;
  filter: drop-shadow(0 6px 18px rgba(70, 50, 120, 0.18));
}

.auth-header-logo-fallback {
  font-size: 1.8rem;
  font-weight: 800;
  color: var(--auth-ink);
}

.auth-header-title {
  font-size: clamp(1.6rem, 3vw, 2.4rem);
  font-weight: 800;
  letter-spacing: -0.02em;
  margin: 0;
  background: linear-gradient(
    100deg,
    var(--auth-primary-dark) 0%,
    var(--auth-primary) 55%,
    var(--auth-accent) 100%
  );
  -webkit-background-clip: text;
  background-clip: text;
  -webkit-text-fill-color: transparent;
}

.auth-header-subtitle {
  font-size: 1rem;
  font-weight: 600;
  color: var(--auth-muted);
  margin: 0.15rem 0 0;
  display: flex;
  align-items: center;
  gap: 0.6rem;
}

.auth-header-subtitle span {
  color: var(--auth-accent);
  font-size: 0.9rem;
}

/* -------------------------------------------------------------------------
   Floating decorative stat cards
   ------------------------------------------------------------------------- */
.auth-float-card {
  position: fixed;
  top: 90px;
  display: flex;
  align-items: center;
  gap: 0.75rem;
  padding: 0.9rem 1.1rem;
  background: var(--auth-glass);
  border: 1px solid var(--auth-border);
  border-radius: 18px;
  box-shadow: var(--auth-shadow);
  backdrop-filter: blur(8px);
  -webkit-backdrop-filter: blur(8px);
  z-index: 5;
  animation: auth-float 5s ease-in-out infinite;
  max-width: 190px;
}

.auth-float-card--left { left: 2vw; }
.auth-float-card--right { right: 2vw; animation-delay: 1.2s; }
.auth-float-card--bottom-right {
  top: auto;
  bottom: 12vh;
  right: 1vw;
  animation-delay: 2.1s;
}

@keyframes auth-float {
  0%, 100% { transform: translateY(0px); }
  50% { transform: translateY(-10px); }
}

.auth-float-icon {
  width: 40px;
  height: 40px;
  min-width: 40px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1.3rem;
  border-radius: 12px;
  background: rgba(124, 111, 232, 0.14);
}

.auth-float-label {
  font-size: 0.72rem;
  color: var(--auth-muted);
  font-weight: 600;
}

.auth-float-value {
  font-size: 1.15rem;
  font-weight: 800;
  color: var(--auth-ink);
  line-height: 1.2;
}

.auth-float-tag {
  font-size: 0.72rem;
  color: #2f9e6e;
  font-weight: 600;
}

@media (max-width: 1100px) {
  .auth-float-card { display: none; }
}

/* -------------------------------------------------------------------------
   Glass card panels (left = feature panel, right = auth form panel)
   Targeted via :has() on the marker div injected as first child, since
   Streamlit's container(border=True) wrapper doesn't accept custom
   classes directly.
   ------------------------------------------------------------------------- */
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-left-marker),
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) {
  background: var(--auth-glass) !important;
  border: 1px solid var(--auth-border) !important;
  border-radius: var(--auth-radius) !important;
  box-shadow: var(--auth-shadow), inset 0 1px 0 rgba(255, 255, 255, 0.6) !important;
  backdrop-filter: blur(14px) !important;
  -webkit-backdrop-filter: blur(14px) !important;
  padding: 0.5rem !important;
  animation: auth-panel-in 0.6s ease-out both;
  transition: transform 0.35s ease, box-shadow 0.35s ease !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) {
  background: var(--auth-glass-strong) !important;
  animation-delay: 0.1s;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-left-marker):hover,
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker):hover {
  transform: translateY(-4px) !important;
  box-shadow: 0 28px 72px rgba(70, 50, 120, 0.26), inset 0 1px 0 rgba(255, 255, 255, 0.7) !important;
}

@keyframes auth-panel-in {
  from {
    opacity: 0;
    transform: translateY(18px);
  }
  to {
    opacity: 1;
    transform: translateY(0);
  }
}

/* -------------------------------------------------------------------------
   Left panel: feature / marketing content
   ------------------------------------------------------------------------- */
.auth-left-content {
  padding: 1.5rem 1.75rem 2rem;
}

.auth-left-badge {
  display: inline-block;
  padding: 0.4rem 1rem;
  font-size: 0.75rem;
  font-weight: 700;
  color: var(--auth-primary-dark);
  background: rgba(124, 111, 232, 0.12);
  border: 1px solid rgba(124, 111, 232, 0.18);
  border-radius: 999px;
  letter-spacing: 0.03em;
  margin-bottom: 1.1rem;
}

.auth-left-title {
  font-size: clamp(1.7rem, 2.5vw, 2.2rem);
  font-weight: 800;
  color: var(--auth-ink);
  line-height: 1.22;
  margin: 0 0 0.85rem;
  letter-spacing: -0.02em;
}

.auth-left-desc {
  font-size: 0.96rem;
  color: var(--auth-muted);
  line-height: 1.65;
  margin: 0 0 1.75rem;
}

.auth-feature-list {
  display: flex;
  flex-direction: column;
  gap: 1rem;
}

.auth-feature-item {
  display: flex;
  align-items: flex-start;
  gap: 0.85rem;
  padding: 0.35rem 0.35rem 0.9rem;
  border-bottom: 1px solid rgba(124, 111, 232, 0.12);
  border-radius: 12px;
  transition: background 0.2s ease, transform 0.2s ease;
}

.auth-feature-item:hover {
  background: rgba(124, 111, 232, 0.06);
  transform: translateX(3px);
}

.auth-feature-item:last-child {
  border-bottom: none;
  padding-bottom: 0;
}

.auth-feature-icon {
  width: 42px;
  height: 42px;
  min-width: 42px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1.25rem;
  border-radius: 13px;
  background: rgba(124, 111, 232, 0.14);
  box-shadow: inset 0 0 0 1px rgba(124, 111, 232, 0.1);
}

.auth-feature-title {
  font-size: 0.95rem;
  font-weight: 700;
  color: var(--auth-ink);
}

.auth-feature-desc {
  font-size: 0.82rem;
  color: var(--auth-muted);
  line-height: 1.45;
}

/* -------------------------------------------------------------------------
   Right panel: auth form content
   ------------------------------------------------------------------------- */
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) {
  padding: 1.5rem 1.75rem 2rem !important;
}

.auth-heading {
  font-size: 1.5rem;
  font-weight: 800;
  color: var(--auth-ink);
  margin-bottom: 0.25rem;
}

.auth-subtitle {
  font-size: 0.9rem;
  color: var(--auth-muted);
  margin-bottom: 1.25rem;
}

.auth-divider {
  display: flex;
  align-items: center;
  text-align: center;
  color: var(--auth-muted);
  font-size: 0.8rem;
  font-weight: 600;
  margin: 1rem 0;
}

.auth-divider::before,
.auth-divider::after {
  content: "";
  flex: 1;
  border-bottom: 1px solid rgba(124, 111, 232, 0.18);
}

.auth-divider span {
  padding: 0 0.75rem;
}

/* -------------------------------------------------------------------------
   Inputs
   ------------------------------------------------------------------------- */
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) label p {
  font-weight: 600 !important;
  color: var(--auth-ink) !important;
  font-size: 0.85rem !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) input {
  border-radius: 12px !important;
  border: 1px solid rgba(124, 111, 232, 0.25) !important;
  background: rgba(255, 255, 255, 0.9) !important;
  padding: 0.65rem 0.9rem !important;
  font-size: 0.95rem !important;
  box-shadow: none !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) input:focus {
  border-color: var(--auth-primary) !important;
  box-shadow: 0 0 0 3px rgba(124, 111, 232, 0.18) !important;
}

/* -------------------------------------------------------------------------
   Buttons
   Primary (form submit) = solid gradient pill CTA.
   Secondary (plain st.button, e.g. "Sign up" / "Forgot password?" /
   "Back to login") = ghost/outline style so they read as lower-emphasis.
   Both targeted globally (not through the broken markdown-wrapper
   pattern) with !important, matching the fix used on the landing page.
   ------------------------------------------------------------------------- */
div[data-testid="stFormSubmitButton"] button,
div[data-testid="stFormSubmitButton"] button[kind="primary"] {
  width: 100% !important;
  padding: 0.9rem 1.5rem !important;
  font-size: 1.02rem !important;
  font-weight: 700 !important;
  letter-spacing: 0.01em !important;
  color: #ffffff !important;
  background: linear-gradient(
    100deg,
    var(--auth-primary-dark) 0%,
    var(--auth-primary) 55%,
    var(--auth-accent) 100%
  ) !important;
  border: none !important;
  border-radius: 999px !important;
  box-shadow: 0 14px 34px rgba(91, 79, 196, 0.45) !important;
  transition: transform 0.22s ease, box-shadow 0.22s ease, filter 0.22s ease !important;
}

div[data-testid="stFormSubmitButton"] button p {
  color: #ffffff !important;
  font-weight: 700 !important;
}

div[data-testid="stFormSubmitButton"] button:hover {
  transform: translateY(-3px) scale(1.01);
  box-shadow: 0 18px 42px rgba(91, 79, 196, 0.55) !important;
  filter: brightness(1.06);
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) div[data-testid="stButton"] button,
div[data-testid="stButton"] button {
  width: 100% !important;
  background: transparent !important;
  color: var(--auth-primary-dark) !important;
  border: none !important;
  border-radius: 10px !important;
  font-weight: 700 !important;
  font-size: 0.92rem !important;
  box-shadow: none !important;
  padding: 0.5rem 0.5rem !important;
  transition: color 0.2s ease, background 0.2s ease !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) div[data-testid="stButton"] button p,
div[data-testid="stButton"] button p {
  color: var(--auth-primary-dark) !important;
  font-weight: 700 !important;
  margin: 0 !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) div[data-testid="stButton"] button:hover,
div[data-testid="stButton"] button:hover {
  background: rgba(124, 111, 232, 0.1) !important;
  color: var(--auth-primary) !important;
  transform: none !important;
}

/* -------------------------------------------------------------------------
   Radio (signup role picker)
   ------------------------------------------------------------------------- */
[data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) [data-testid="stRadio"] label p {
  font-size: 0.85rem !important;
}

/* -------------------------------------------------------------------------
   Mockup preview panel (decorative only -- donut, trend chart, chat bubble)
   ------------------------------------------------------------------------- */
.auth-mockup {
  display: flex;
  flex-direction: column;
  gap: 0.9rem;
  padding: 1.5rem 1.25rem 1.5rem 0;
  position: relative;
}

.auth-mockup-card {
  background: rgba(255, 255, 255, 0.85);
  border: 1px solid rgba(124, 111, 232, 0.14);
  border-radius: 16px;
  padding: 0.9rem 1rem;
  box-shadow: 0 8px 24px rgba(70, 50, 120, 0.1);
}

.auth-mockup-title {
  font-size: 0.75rem;
  font-weight: 700;
  color: var(--auth-muted);
  margin-bottom: 0.5rem;
}

.auth-mockup-donut-row {
  display: flex;
  align-items: center;
  gap: 0.75rem;
}

.auth-mockup-donut {
  width: 58px;
  height: 58px;
  min-width: 58px;
  border-radius: 50%;
  background: conic-gradient(
    var(--auth-primary) 0deg 302deg,
    rgba(124, 111, 232, 0.15) 302deg 360deg
  );
  display: flex;
  align-items: center;
  justify-content: center;
}

.auth-mockup-donut-inner {
  width: 42px;
  height: 42px;
  border-radius: 50%;
  background: #ffffff;
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
}

.auth-mockup-donut-value {
  font-size: 0.8rem;
  font-weight: 800;
  color: var(--auth-ink);
  line-height: 1;
}

.auth-mockup-donut-label {
  font-size: 0.5rem;
  color: var(--auth-muted);
  font-weight: 600;
}

.auth-mockup-spark {
  flex: 1;
  height: 40px;
}

.auth-mockup-spark--wide {
  width: 100%;
  height: 56px;
}

.auth-mockup-axis {
  font-size: 0.6rem;
  color: var(--auth-muted);
  display: flex;
  justify-content: space-between;
  margin-top: 0.35rem;
}

.auth-mockup-chat {
  display: flex;
  align-items: flex-end;
  gap: 0.5rem;
  align-self: flex-end;
}

.auth-mockup-bubble {
  background: rgba(255, 255, 255, 0.92);
  border: 1px solid rgba(124, 111, 232, 0.16);
  border-radius: 14px 14px 4px 14px;
  padding: 0.6rem 0.8rem;
  font-size: 0.72rem;
  color: var(--auth-ink);
  line-height: 1.4;
  box-shadow: 0 8px 20px rgba(70, 50, 120, 0.1);
  max-width: 150px;
}

.auth-mockup-avatar {
  width: 42px;
  height: 42px;
  min-width: 42px;
  border-radius: 50%;
  background: linear-gradient(135deg, var(--auth-primary) 0%, var(--auth-accent) 100%);
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1.2rem;
  box-shadow: 0 8px 20px rgba(91, 79, 196, 0.3);
}

@media (max-width: 900px) {
  .auth-mockup { display: none; }
}

/* -------------------------------------------------------------------------
   Forgot password / Create account links
   ------------------------------------------------------------------------- */
.auth-forgot-link div[data-testid="stButton"] button {
  text-align: right !important;
  justify-content: flex-end !important;
}

.auth-create-account div[data-testid="stButton"] button {
  font-size: 0.85rem !important;
  font-weight: 500 !important;
  color: var(--auth-muted) !important;
}

.auth-create-account div[data-testid="stButton"] button p {
  color: var(--auth-muted) !important;
  font-weight: 500 !important;
}

/* -------------------------------------------------------------------------
   Footer trust bar
   ------------------------------------------------------------------------- */
.auth-footer-bar {
  width: 100%;
  margin-top: 1.5rem;
  padding: 0.85rem 1.5rem;
  display: flex;
  flex-wrap: wrap;
  align-items: center;
  justify-content: center;
  gap: 0.6rem;
  background: rgba(255, 255, 255, 0.7);
  border: 1px solid rgba(255, 255, 255, 0.9);
  border-radius: 999px;
  box-shadow: 0 8px 24px rgba(70, 50, 120, 0.1);
  backdrop-filter: blur(8px);
  -webkit-backdrop-filter: blur(8px);
  font-size: 0.8rem;
  color: var(--auth-muted);
  font-weight: 500;
}

.auth-footer-sep {
  color: rgba(107, 101, 132, 0.35);
}

.auth-footer-highlight {
  color: var(--auth-primary-dark);
  font-weight: 700;
}

@media (max-width: 900px) {
  .auth-footer-bar {
    flex-direction: column;
    border-radius: 20px;
    text-align: center;
  }
  .auth-footer-sep { display: none; }
}

/* -------------------------------------------------------------------------
   Responsive
   ------------------------------------------------------------------------- */
@media (max-width: 900px) {
  .auth-left-content,
  [data-testid="stVerticalBlockBorderWrapper"]:has(.auth-right-marker) {
    padding: 1.25rem !important;
  }
  .block-container {
    padding-top: 2rem !important;
  }
}

Overwriting styles/auth.css


In [ ]:
%%writefile components/sidebar.py
"""
components/sidebar.py

Premium sidebar for MoodMentor's authenticated app screens (Home, Journal,
Wellness Chat, Face Detection, Relax, Dashboard, Reports).

IMPORTANT: UI ONLY. Reuses the exact session_state keys and logout logic
from the original inline sidebar block in app.py:
    - st.session_state.nav
    - st.session_state.token / page / show_auth_panel (cleared on logout)
No new backend calls. Nav options are only ever what the caller passes in
(app.py already computes these correctly per role) -- this module does not
invent extra nav items.

Usage in app.py (replaces the `with st.sidebar: ...` block):

    from components.sidebar import render_sidebar
    ...
    if role == "employee":
        nav_options = ["Home", "Journal", "Wellness Chat", "Face Detection", "Relax", "Dashboard"]
    else:
        nav_options = ["Reports"]
    render_sidebar(user, role, nav_options)
"""

import base64
from pathlib import Path

import streamlit as st

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _load_dashboard_css() -> bool:
    css_path = _resolve_first(
        ("styles", "dashboard.css"),
        ("assets", "styles", "dashboard.css"),
        ("assets", "css", "dashboard.css"),
    )
    if css_path.exists():
        st.markdown(f"<style>{css_path.read_text(encoding='utf-8')}</style>", unsafe_allow_html=True)
        return True
    return False


# Only icons for routes that actually exist in nav_options get used --
# this dict is a lookup table, not a feature list.
_NAV_ICONS = {
    "Home": "🏠",
    "Journal": "📖",
    "Wellness Chat": "💬",
    "Face Detection": "📷",
    "Relax": "🧘",
    "Dashboard": "📊",
    "Reports": "📈",
}


def render_sidebar(user: dict, role: str, nav_options: list, debug: bool = True) -> None:
    """Render the styled sidebar and update st.session_state.nav.

    `nav_options` must be exactly what app.py already computes for the
    current role -- this function does not add or remove routes.

    `debug=True` (default) shows a small warning in the sidebar itself if
    dashboard.css can't be found, listing the exact paths checked. Set to
    False once you've confirmed the file is in place.
    """
    css_ok = _load_dashboard_css()

    bg_path = _resolve_first(("assets", "backgrounds", "background.png"))
    bg_b64 = _get_base64_of_file(str(bg_path))

    with st.sidebar:
        if debug and not css_ok:
            checked = [
                str(Path(root, "styles", "dashboard.css")) for root in _CANDIDATE_ROOTS
            ] + [
                str(Path(root, "assets", "styles", "dashboard.css")) for root in _CANDIDATE_ROOTS
            ]
            st.warning(
                "dashboard.css not found. Sidebar/Home will look unstyled "
                "until it exists at one of:\n\n"
                + "\n\n".join(f"- `{p}`" for p in checked[:3])
            )

        logo_path = _resolve_first(("assets", "logo", "logo.png"))
        logo_b64 = _get_base64_of_file(str(logo_path))
        if logo_b64:
            logo_html = f'<img src="data:image/png;base64,{logo_b64}" class="mm-sidebar-logo-img" alt="Logo" />'
        else:
            logo_html = '<div class="mm-sidebar-brand-icon">🌿</div>'

        st.markdown(
            f"""
            <div class="mm-sidebar-brand">
                {logo_html}
            </div>
            """,
            unsafe_allow_html=True,
        )

        labeled_options = [f"{_NAV_ICONS.get(o, '•')}  {o}" for o in nav_options]
        current = st.session_state.nav if st.session_state.nav in nav_options else nav_options[0]
        current_label = f"{_NAV_ICONS.get(current, '•')}  {current}"
        default_index = labeled_options.index(current_label) if current_label in labeled_options else 0

        choice_label = st.radio(
            "Navigate", labeled_options, index=default_index,
            label_visibility="collapsed", key="sidebar_nav_radio",
        )
        st.session_state.nav = nav_options[labeled_options.index(choice_label)]

        initials = "".join(w[0] for w in user["username"].split()[:2]).upper() or "U"
        st.markdown(
            f"""
            <div class="mm-sidebar-profile">
                <div class="mm-sidebar-avatar">{initials}</div>
                <div>
                    <div class="mm-sidebar-username">{user['username']}</div>
                    <div class="mm-sidebar-role">{role.capitalize()}</div>
                </div>
            </div>
            <div class="mm-sidebar-email">{user['email']}</div>
            """,
            unsafe_allow_html=True,
        )

        if st.button("↩  Log out", use_container_width=True, key="sidebar_logout_btn"):
            st.session_state.token = None
            st.session_state.page = "welcome"
            st.session_state.show_auth_panel = False
            st.rerun()

        if bg_b64:
            st.markdown(
                f"""
                <style>
                section[data-testid="stSidebar"] {{
                    background-image:
                        linear-gradient(180deg, rgba(247,245,255,0.98) 0%, rgba(247,245,255,0.55) 30%, rgba(247,245,255,0.02) 55%, rgba(247,245,255,0) 100%),
                        url("data:image/png;base64,{bg_b64}") !important;
                    background-size: cover !important;
                    background-position: bottom center !important;
                    background-repeat: no-repeat !important;
                    background-attachment: fixed !important;
                }}
                section[data-testid="stSidebar"] > div:first-child {{
                    background: transparent !important;
                }}
                </style>
                """,
                unsafe_allow_html=True,
            )

Overwriting components/sidebar.py


In [ ]:
%%writefile components/home.py
"""
components/home.py

Premium "Home" dashboard section for MoodMentor, matching the reference
design: metric cards, mood picker, calendar, and a right-hand info column.

IMPORTANT: UI ONLY for the core dashboard (metrics/mood-picker/calendar) --
every db call, every session_state key (picked_mood, today_mood_saved,
cal_year, cal_month), and every condition is copied verbatim from the
original inline Home section in app.py.

The right-hand column (Daily Affirmation, Take a Breath, Insights, Tip of
the Day) is NEW and not from the original code, scoped exactly as agreed:
    - Daily Affirmation: decorative only (static rotating quote)
    - Take a Breath: REAL -- switches st.session_state.nav to "Relax"
      (your actual Relax section) and reruns, same mechanism the app
      already uses elsewhere
    - Insights: decorative only (no fabricated data claims)
    - Tip of the Day: decorative only (static rotating tip)

Usage in app.py (replaces the original `if section == "Home": ...` body):

    from components.home import render_home_section
    ...
    if section == "Home":
        render_home_section(user)
"""

import base64
import calendar
from datetime import date, datetime
from pathlib import Path

import streamlit as st

from db import MOOD_LABELS, MOOD_EMOJI, get_mood_logs_for_month, get_user_mood_history, save_manual_mood

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


# --------------------------------------------------------------------------- #
# Duplicated (not imported) from app.py to avoid a circular import --
# app.py imports this module, so this module can't import back from app.py.
# Keep these in sync with app.py's MOOD_STYLE if you change mood colors.
# --------------------------------------------------------------------------- #

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#2ecc71"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#3498db"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#e67e22"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}


def _style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})


# --------------------------------------------------------------------------- #
# Decorative-only static content (clearly not derived from real data)
# --------------------------------------------------------------------------- #

_AFFIRMATIONS = [
    "You are stronger than you think and braver than you feel.",
    "Every feeling is valid. Every day is a new beginning.",
    "Small steps still move you forward.",
    "You don't have to be perfect to be proud of yourself.",
    "Rest is productive too.",
    "Your feelings are data, not a verdict.",
    "Progress, not perfection.",
]

_TIPS = [
    "Go for a short walk in nature. It boosts your mood instantly.",
    "Drink a glass of water and stretch for 60 seconds.",
    "Write down one thing you're grateful for today.",
    "Step away from your screen for 5 minutes.",
    "Send a kind message to someone you appreciate.",
    "Take three slow, deep breaths before your next meeting.",
]


def _pick_of_the_day(options: list) -> str:
    """Deterministic per-day rotation -- same value all day, changes daily."""
    idx = date.today().toordinal() % len(options)
    return options[idx]


# --------------------------------------------------------------------------- #
# Metric tile
# --------------------------------------------------------------------------- #

def _metric_tile(label: str, icon: str, value: str, sub: str, accent: str) -> None:
    st.markdown(
        f"""
        <div class="mm-home-metric" style="--accent:{accent}">
            <div class="mm-home-metric-label">{label}</div>
            <div class="mm-home-metric-row">
                <div class="mm-home-metric-icon">{icon}</div>
                <div>
                    <div class="mm-home-metric-value">{value}</div>
                    <div class="mm-home-metric-sub">{sub}</div>
                </div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


# --------------------------------------------------------------------------- #
# Right column (decorative + one real feature)
# --------------------------------------------------------------------------- #

def _render_right_column() -> None:
    quote = _pick_of_the_day(_AFFIRMATIONS)
    tip = _pick_of_the_day(_TIPS)

    bg_path = _resolve_first(
        ("assets", "backgrounds", "affirmation-bg.png"),
        ("assets", "backgrounds", "background.png"),
    )
    bg_b64 = _get_base64_of_file(str(bg_path))
    bg_style = (
        f'background-image: linear-gradient(180deg, rgba(91,79,196,0.25) 0%, rgba(60,45,140,0.65) 100%), '
        f'url("data:image/png;base64,{bg_b64}"); background-size: cover; background-position: center;'
        if bg_b64 else ""
    )

    st.markdown(
        f"""
        <div class="mm-home-side-card mm-home-affirmation" style='{bg_style}'>
            <div class="mm-home-side-title">💬 Daily Affirmation</div>
            <div class="mm-home-affirmation-text">&ldquo;{quote}&rdquo;</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    breath_img_path = _resolve_first(("assets", "illustrations", "breathing-woman.png"))
    breath_img_b64 = _get_base64_of_file(str(breath_img_path))
    if breath_img_b64:
        breath_illustration_html = (
            f'<img src="data:image/png;base64,{breath_img_b64}" '
            f'class="mm-home-breath-img" alt="Meditation illustration" />'
        )
    else:
        breath_illustration_html = '<div class="mm-home-breath-illustration">🧘‍♀️</div>'

    st.markdown(
        f"""
        <div class="mm-home-side-card mm-home-breath-card">
            <div class="mm-home-side-title">🌬️ Take a Breath</div>
            <div class="mm-home-breath-row">
                <div class="mm-home-side-desc">Take a 2-minute mindful breathing break.</div>
                {breath_illustration_html}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )
    if st.button("▶  Start Breathing", key="home_start_breathing_btn", use_container_width=True):
        st.session_state.nav = "Relax"
        st.rerun()

    st.markdown(
        """
        <div class="mm-home-side-card">
            <div class="mm-home-side-title">📈 Insights</div>
            <div class="mm-home-side-desc">Check your Dashboard tab for mood trends and detailed analytics.</div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        f"""
        <div class="mm-home-side-card mm-home-tip-card">
            <div class="mm-home-side-title">💡 Tip of the Day</div>
            <div class="mm-home-side-desc">{tip}</div>
            <div class="mm-home-tip-leaf">🌿</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


# --------------------------------------------------------------------------- #
# Public entry point
# --------------------------------------------------------------------------- #

def render_home_section(user: dict) -> None:
    """Render the Home dashboard. Identical data/logic to the original
    inline `if section == "Home":` block -- only the layout/markup and the
    (clearly-scoped) right column are new."""

    greeting = "Good Morning" if datetime.now().hour < 12 else (
        "Good Afternoon" if datetime.now().hour < 18 else "Good Evening")
    now = datetime.now()

    _bird_svg = (
        '<svg viewBox="0 0 24 12" xmlns="http://www.w3.org/2000/svg">'
        '<path d="M1 9 Q6 1 12 9 Q18 1 23 9" stroke="#7c6fe8" stroke-width="2" '
        'fill="none" stroke-linecap="round"/></svg>'
    )
    st.markdown(
        f"""
        <div class="mm-home-decor mm-home-decor--branch">🍃🌿🍃</div>
        <div class="mm-home-decor mm-home-decor--bird1">{_bird_svg}</div>
        <div class="mm-home-decor mm-home-decor--bird2">{_bird_svg}</div>
        <div class="mm-home-decor mm-home-decor--bird3">{_bird_svg}</div>
        <div class="mm-home-decor mm-home-decor--floral">🌸🌷🌼</div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        f"""
        <div class="mm-home-header">
            <div>
                <h2 class="mm-home-greeting">{greeting}, <span>{user['username']}</span>!</h2>
                <p class="mm-home-subgreeting">Every feeling is valid. Every day is a new beginning. 💜</p>
            </div>
            <div class="mm-home-datetime">
                📅 {now.strftime('%A, %-d %B %Y')} &nbsp;|&nbsp; 🕐 {now.strftime('%I:%M %p')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    main_col, side_col = st.columns([2.3, 1], gap="large")

    with main_col:
        history_all = get_user_mood_history(user["id"], limit=500)
        latest = history_all[0] if history_all else None
        today_count = sum(1 for h in history_all if h["mood_date"] == date.today())
        streak = 0
        day_ptr = date.today()
        day_set = {h["mood_date"] for h in history_all}
        while day_ptr in day_set:
            streak += 1
            day_ptr = date.fromordinal(day_ptr.toordinal() - 1)

        positive_count = sum(1 for h in history_all if h["sentiment"] == "Happy")
        overall_score = int(100 * positive_count / len(history_all)) if history_all else 0

        m1, m2, m3, m4 = st.columns(4)
        with m1:
            if latest:
                s = _style_for(latest["sentiment"])
                _metric_tile("CURRENT MOOD", s["emoji"], latest["sentiment"], "Keep shining!", s["color"])
            else:
                _metric_tile("CURRENT MOOD", "—", "No data", "Pick a mood below", "#bdbdbd")
        with m2:
            _metric_tile(
                "OVERALL SCORE", "📊", f"{overall_score}%",
                "You're improving!" if overall_score >= 50 else "Needs care",
                "#2ecc71" if overall_score >= 50 else "#e67e22",
            )
        with m3:
            _metric_tile("ENTRIES TODAY", "📝", str(today_count), "Keep it up!", "#7c6fe8")
        with m4:
            _metric_tile("CURRENT STREAK", "🔥", f"{streak} Days", "Great consistency!", "#f1a33d")

        st.markdown('<div class="mm-home-section-gap"></div>', unsafe_allow_html=True)
        st.markdown('<h3 class="mm-home-section-title">How Do You Feel Today?</h3>', unsafe_allow_html=True)
        st.caption("Select your current mood")

        cols = st.columns(len(MOOD_LABELS))
        picked = st.session_state.get("picked_mood")
        for col, label in zip(cols, MOOD_LABELS):
            s = _style_for(label)
            is_picked = picked == label
            with col:
                st.markdown(
                    f"""
                    <div class="mm-mood-card {'mm-mood-card--active' if is_picked else ''}" style="--mood-color:{s['color']}">
                        <div class="mm-mood-emoji">{s['emoji']}</div>
                        <div class="mm-mood-label">{label}</div>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )
                if st.button("Select", key=f"pick_{label}", use_container_width=True, help=f"Select {label}"):
                    st.session_state.picked_mood = label
                    st.rerun()

        st.write("")
        confirm_col = st.columns([2, 1.2, 2])[1]
        with confirm_col:
            disabled = picked is None
            if st.button("♥  Save Mood", type="primary", disabled=disabled, use_container_width=True):
                save_manual_mood(user["id"], st.session_state.picked_mood)
                st.session_state.today_mood_saved = True
                st.session_state.picked_mood = None
                st.rerun()

        if st.session_state.today_mood_saved:
            st.success("Today's mood saved!")
            st.session_state.today_mood_saved = False

        st.markdown('<div class="mm-home-section-gap"></div>', unsafe_allow_html=True)
        st.markdown('<h3 class="mm-home-section-title">Your Mood Calendar</h3>', unsafe_allow_html=True)
        st.caption("Track your emotional journey")

        nav_l, nav_mid, nav_r = st.columns([1, 3, 1])
        if nav_l.button("← Prev"):
            m, y = st.session_state.cal_month - 1, st.session_state.cal_year
            if m == 0: m, y = 12, y - 1
            st.session_state.cal_month, st.session_state.cal_year = m, y
            st.rerun()
        if nav_r.button("Next →"):
            m, y = st.session_state.cal_month + 1, st.session_state.cal_year
            if m == 13: m, y = 1, y + 1
            st.session_state.cal_month, st.session_state.cal_year = m, y
            st.rerun()
        nav_mid.markdown(
            f"<h4 style='text-align:center'>{calendar.month_name[st.session_state.cal_month]} "
            f"{st.session_state.cal_year}</h4>", unsafe_allow_html=True,
        )

        logs = get_mood_logs_for_month(user["id"], st.session_state.cal_year, st.session_state.cal_month)
        by_day = {row["mood_date"].day: row for row in logs}

        weeks = calendar.Calendar(firstweekday=6).monthdayscalendar(
            st.session_state.cal_year, st.session_state.cal_month
        )
        day_names = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
        header_cols = st.columns(7)
        for c, name in zip(header_cols, day_names):
            c.markdown(f"<div class='mm-cal-dayname'>{name}</div>", unsafe_allow_html=True)

        for week in weeks:
            cols = st.columns(7)
            for col, day_num in zip(cols, week):
                if day_num == 0:
                    col.write("")
                    continue
                entry = by_day.get(day_num)
                s = _style_for(entry["sentiment"] if entry else None)
                is_today = date(st.session_state.cal_year, st.session_state.cal_month, day_num) == date.today()
                col.markdown(
                    f"""
                    <div class="mm-cal-cell {'mm-cal-cell--today' if is_today else ''}"
                         style="--cal-color:{s['color']}">
                        <div class="mm-cal-daynum">{day_num}</div>
                        <div class="mm-cal-emoji">{s['emoji'] or '·'}</div>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )

        legend_html = "".join(
            f'<span class="mm-cal-legend-item"><span class="mm-cal-legend-dot" '
            f'style="background:{_style_for(l)["color"]}"></span>{_style_for(l)["emoji"]} {l} '
            f'{sum(1 for h in history_all if h["sentiment"] == l)}</span>'
            for l in MOOD_LABELS
        )
        st.markdown(f'<div class="mm-cal-legend">{legend_html}</div>', unsafe_allow_html=True)

    with side_col:
        _render_right_column()

Overwriting components/home.py


In [ ]:
%%writefile styles/dashboard.css
/* ==========================================================================
   styles/dashboard.css
   Sidebar + Home dashboard styles for MoodMentor's authenticated screens.
   Loaded by components/sidebar.py (runs on every authenticated page load,
   so these rules apply regardless of which section is active).
   ========================================================================== */

:root {
  --mm-primary: #7c6fe8;
  --mm-primary-dark: #5b4fc4;
  --mm-ink: #241f3d;
  --mm-muted: #6b6584;
  --mm-card-radius: 20px;
  --mm-shadow: 0 10px 30px rgba(70, 50, 120, 0.08);
}

/* Override app.py's original grey-blue .stApp gradient (set in inject_css())
   with a warmer off-white/lavender tone matching the reference. This file
   loads after inject_css() runs, so !important + later source order wins. */
.stApp {
  background: linear-gradient(135deg, #fdfbff 0%, #f5f2ff 45%, #ece6fb 100%) !important;
}

/* -------------------------------------------------------------------------
   Sidebar shell
   ------------------------------------------------------------------------- */
section[data-testid="stSidebar"] {
  border-right: 1px solid rgba(124, 111, 232, 0.1) !important;
}

section[data-testid="stSidebar"] * {
  color: var(--mm-ink) !important;
}

.mm-sidebar-brand {
  display: flex;
  align-items: center;
  gap: 0.6rem;
  padding: 0.5rem 0.25rem 1.25rem;
}

.mm-sidebar-brand img.mm-sidebar-logo-img {
  max-width: 190px;
}

.mm-sidebar-brand-icon {
  width: 40px;
  height: 40px;
  min-width: 40px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1.3rem;
  border-radius: 12px;
  background: rgba(124, 111, 232, 0.14);
}

.mm-sidebar-brand-name {
  font-size: 1rem;
  font-weight: 800 !important;
  color: var(--mm-ink) !important;
}

.mm-sidebar-brand-tag {
  font-size: 0.7rem !important;
  color: var(--mm-muted) !important;
  font-weight: 500 !important;
}

.mm-sidebar-logo-img {
  max-width: 100%;
  height: auto;
  display: block;
}

/* Nav radio -> styled as vertical icon+label rows.
   Multiple selector strategies are stacked here because Streamlit's exact
   DOM/attributes for radio "checked" state vary by version -- data-checked
   works on some, :has(input:checked) is a modern-browser-safe fallback. */
section[data-testid="stSidebar"] div[role="radiogroup"] {
  gap: 0.3rem;
  display: flex;
  flex-direction: column;
}

section[data-testid="stSidebar"] div[role="radiogroup"] > label {
  background: transparent !important;
  padding: 0.7rem 0.9rem !important;
  border-radius: 12px !important;
  border: none !important;
  box-shadow: none !important;
  transition: background 0.2s ease !important;
  font-weight: 600 !important;
  display: flex !important;
  align-items: center !important;
  cursor: pointer;
}

section[data-testid="stSidebar"] div[role="radiogroup"] > label:hover {
  background: rgba(124, 111, 232, 0.08) !important;
}

section[data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"],
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) {
  background: var(--mm-primary) !important;
  box-shadow: 0 6px 18px rgba(124, 111, 232, 0.35) !important;
}

section[data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] p,
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) p,
section[data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] div,
section[data-testid="stSidebar"] div[role="radiogroup"] > label:has(input:checked) div {
  color: #ffffff !important;
}

/* Hide the native radio circle -- the colored row background is the
   selection indicator instead. Sized to 0 rather than display:none so it
   stays in the accessibility/click target. */
section[data-testid="stSidebar"] div[role="radiogroup"] label > div:first-child {
  width: 0 !important;
  height: 0 !important;
  min-width: 0 !important;
  opacity: 0 !important;
  overflow: hidden !important;
  margin: 0 !important;
}

/* -------------------------------------------------------------------------
   Profile card
   ------------------------------------------------------------------------- */
.mm-sidebar-profile {
  display: flex;
  align-items: center;
  gap: 0.7rem;
  margin-top: 1.5rem;
  padding: 0.9rem;
  background: rgba(255, 255, 255, 0.94);
  border: 1px solid rgba(124, 111, 232, 0.14);
  border-radius: 16px 16px 0 0;
  box-shadow: 0 -4px 16px rgba(70, 50, 120, 0.06);
}

.mm-sidebar-avatar {
  width: 42px;
  height: 42px;
  min-width: 42px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: linear-gradient(135deg, var(--mm-primary) 0%, #a78bfa 100%);
  color: #ffffff !important;
  font-weight: 800;
  font-size: 0.9rem;
}

.mm-sidebar-username {
  font-weight: 700 !important;
  font-size: 0.92rem;
}

.mm-sidebar-role {
  font-size: 0.72rem !important;
  color: var(--mm-muted) !important;
}

.mm-sidebar-email {
  font-size: 0.72rem !important;
  color: var(--mm-muted) !important;
  background: rgba(255, 255, 255, 0.94);
  padding: 0.5rem 0.9rem;
  word-break: break-all;
}

section[data-testid="stSidebar"] div[data-testid="stButton"] button {
  background: rgba(255, 255, 255, 0.94) !important;
  color: var(--mm-primary-dark) !important;
  border: 1px solid rgba(124, 111, 232, 0.25) !important;
  border-radius: 0 0 14px 14px !important;
  font-weight: 700 !important;
  box-shadow: 0 6px 16px rgba(70, 50, 120, 0.1) !important;
}

section[data-testid="stSidebar"] div[data-testid="stButton"] button p {
  color: var(--mm-primary-dark) !important;
}

/* -------------------------------------------------------------------------
   Decorative flourishes (matching the reference's illustrated feel)
   ------------------------------------------------------------------------- */
.mm-home-decor {
  position: fixed;
  pointer-events: none;
  z-index: 1;
  opacity: 0.75;
  filter: drop-shadow(0 2px 6px rgba(70, 50, 120, 0.1));
}

.mm-home-decor--branch {
  top: 0.5vh;
  right: 3vw;
  font-size: 2.2rem;
  transform: rotate(8deg);
}

.mm-home-decor--bird1 { top: 4vh; right: 22vw; animation: mm-bird-float 6s ease-in-out infinite; }
.mm-home-decor--bird2 { top: 8vh; right: 28vw; animation: mm-bird-float 7s ease-in-out infinite 1s; }
.mm-home-decor--bird3 { top: 3vh; right: 34vw; animation: mm-bird-float 5.5s ease-in-out infinite 0.5s; }

.mm-home-decor--bird1 svg, .mm-home-decor--bird2 svg, .mm-home-decor--bird3 svg {
  width: 18px;
  height: 9px;
  display: block;
}

@keyframes mm-bird-float {
  0%, 100% { transform: translateY(0) translateX(0); }
  50% { transform: translateY(-6px) translateX(4px); }
}

.mm-home-decor--floral {
  bottom: 1vh;
  left: 1vw;
  font-size: 1.6rem;
}

@media (max-width: 1100px) {
  .mm-home-decor { display: none; }
}

/* -------------------------------------------------------------------------
   Home: header
   ------------------------------------------------------------------------- */
.mm-home-header {
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  flex-wrap: wrap;
  gap: 1rem;
  margin-bottom: 1.5rem;
}

.mm-home-greeting {
  font-size: 1.9rem !important;
  font-weight: 800 !important;
  color: var(--mm-ink) !important;
  margin: 0 !important;
}

.mm-home-greeting span {
  color: var(--mm-primary);
}

.mm-home-subgreeting {
  color: var(--mm-muted) !important;
  font-size: 0.95rem !important;
  margin: 0.25rem 0 0 !important;
}

.mm-home-datetime {
  background: rgba(255, 255, 255, 0.8);
  border: 1px solid rgba(124, 111, 232, 0.16);
  border-radius: 999px;
  padding: 0.55rem 1.1rem;
  font-size: 0.85rem;
  font-weight: 600;
  color: var(--mm-ink) !important;
  box-shadow: var(--mm-shadow);
  white-space: nowrap;
}

/* -------------------------------------------------------------------------
   Home: metric tiles
   ------------------------------------------------------------------------- */
.mm-home-metric {
  background: rgba(255, 255, 255, 0.85);
  border: 1px solid rgba(124, 111, 232, 0.12);
  border-top: 3px solid var(--accent, var(--mm-primary));
  border-radius: 18px;
  padding: 1.1rem 1.2rem;
  box-shadow: var(--mm-shadow);
  transition: transform 0.2s ease, box-shadow 0.2s ease;
}

.mm-home-metric:hover {
  transform: translateY(-3px);
  box-shadow: 0 16px 36px rgba(70, 50, 120, 0.12);
}

.mm-home-metric-label {
  font-size: 0.68rem;
  font-weight: 800;
  color: var(--accent, var(--mm-primary));
  letter-spacing: 0.04em;
  margin-bottom: 0.6rem;
}

.mm-home-metric-row {
  display: flex;
  align-items: center;
  gap: 0.7rem;
}

.mm-home-metric-icon {
  font-size: 1.6rem;
  width: 46px;
  height: 46px;
  min-width: 46px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: color-mix(in srgb, var(--accent, var(--mm-primary)) 16%, white);
}

.mm-home-metric-value {
  font-size: 1.3rem;
  font-weight: 800;
  color: var(--mm-ink);
  line-height: 1.2;
}

.mm-home-metric-sub {
  font-size: 0.72rem;
  color: var(--mm-muted);
}

.mm-home-section-gap { height: 1.75rem; }

.mm-home-section-title {
  font-size: 1.25rem !important;
  font-weight: 800 !important;
  color: var(--mm-ink) !important;
  margin: 0 0 0.15rem !important;
}

/* -------------------------------------------------------------------------
   Home: mood picker cards
   ------------------------------------------------------------------------- */
.mm-mood-card {
  background: rgba(255, 255, 255, 0.85);
  border: 2px solid rgba(124, 111, 232, 0.12);
  border-radius: 16px;
  padding: 0.9rem 0.5rem 0.6rem;
  text-align: center;
  transition: transform 0.15s ease, border-color 0.15s ease;
}

.mm-mood-card:hover {
  transform: translateY(-2px);
}

.mm-mood-card--active {
  border-color: var(--mood-color) !important;
  background: color-mix(in srgb, var(--mood-color) 10%, white) !important;
}

.mm-mood-emoji {
  font-size: 1.8rem;
  line-height: 1.1;
  width: 54px;
  height: 54px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: color-mix(in srgb, var(--mood-color) 16%, white);
  margin: 0 auto;
  transition: transform 0.2s ease;
}

.mm-mood-card:hover .mm-mood-emoji {
  transform: scale(1.08) rotate(-4deg);
}

.mm-mood-label {
  font-size: 0.8rem;
  font-weight: 700;
  color: var(--mm-ink);
  margin-top: 0.2rem;
}

/* pull the "Select" button visually into the card above it */
div[data-testid="column"]:has(.mm-mood-card) div[data-testid="stButton"] {
  margin-top: -0.4rem;
}

div[data-testid="column"]:has(.mm-mood-card) div[data-testid="stButton"] button {
  border-radius: 0 0 14px 14px !important;
  font-size: 0.72rem !important;
  padding: 0.3rem 0.1rem !important;
  background: rgba(124, 111, 232, 0.06) !important;
  color: var(--mm-primary-dark) !important;
  border: none !important;
  box-shadow: none !important;
  white-space: nowrap !important;
  overflow: hidden !important;
  text-overflow: ellipsis !important;
}

/* -------------------------------------------------------------------------
   Home: mood calendar
   ------------------------------------------------------------------------- */
.mm-cal-dayname {
  text-align: center;
  font-size: 0.75rem;
  font-weight: 700;
  color: var(--mm-muted);
  padding-bottom: 0.4rem;
}

.mm-cal-cell {
  background: rgba(255, 255, 255, 0.6);
  border: 1px solid color-mix(in srgb, var(--cal-color, #ddd) 35%, transparent);
  border-radius: 12px;
  padding: 0.4rem 0.2rem;
  text-align: center;
  margin-bottom: 0.4rem;
}

.mm-cal-cell--today {
  border: 2px solid var(--mm-primary);
  box-shadow: 0 0 0 2px rgba(124, 111, 232, 0.15);
}

.mm-cal-daynum {
  font-size: 0.7rem;
  color: var(--mm-muted);
}

.mm-cal-emoji {
  font-size: 1.3rem;
}

.mm-cal-legend {
  display: flex;
  flex-wrap: wrap;
  gap: 0.9rem;
  margin-top: 0.5rem;
  font-size: 0.78rem;
  color: var(--mm-muted);
}

.mm-cal-legend-item {
  display: inline-flex;
  align-items: center;
  gap: 0.3rem;
}

.mm-cal-legend-dot {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  display: inline-block;
}

/* -------------------------------------------------------------------------
   Home: right column
   ------------------------------------------------------------------------- */
.mm-home-side-card {
  background: rgba(255, 255, 255, 0.85);
  border: 1px solid rgba(124, 111, 232, 0.12);
  border-radius: 20px;
  padding: 1.5rem 1.6rem;
  min-height: 108px;
  box-shadow: var(--mm-shadow);
  margin-bottom: 1.1rem;
}

.mm-home-side-title {
  font-size: 1.05rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-bottom: 0.55rem;
}

.mm-home-side-desc {
  font-size: 0.92rem;
  color: var(--mm-muted);
  line-height: 1.55;
}

.mm-home-breath-row {
  display: flex;
  align-items: flex-end;
  justify-content: space-between;
  gap: 0.5rem;
}

.mm-home-breath-illustration {
  font-size: 3rem;
  line-height: 1;
  filter: drop-shadow(0 4px 10px rgba(70, 50, 120, 0.15));
}

.mm-home-breath-img {
  width: 84px;
  height: auto;
  filter: drop-shadow(0 4px 10px rgba(70, 50, 120, 0.15));
}

.mm-home-affirmation {
  border: none;
  background-size: cover !important;
  background-position: center !important;
  position: relative;
  overflow: hidden;
  min-height: 190px !important;
}

.mm-home-affirmation::after {
  content: "";
  position: absolute;
  inset: 0;
  background: linear-gradient(160deg, rgba(91,79,196,0.15) 0%, rgba(124,111,232,0.05) 100%);
  pointer-events: none;
}

.mm-home-affirmation .mm-home-side-title,
.mm-home-affirmation-text {
  color: #ffffff !important;
  position: relative;
  z-index: 1;
  text-shadow: 0 2px 8px rgba(0, 0, 0, 0.25);
}

.mm-home-affirmation-text {
  font-size: 1.05rem;
  font-style: italic;
  line-height: 1.55;
}

.mm-home-tip-card {
  position: relative;
  overflow: hidden;
}

.mm-home-tip-leaf {
  position: absolute;
  bottom: -0.4rem;
  right: 0.3rem;
  font-size: 2.6rem;
  opacity: 0.55;
  transform: rotate(-12deg);
  pointer-events: none;
}

/* ==========================================================================
   Journal page
   ========================================================================== */

.mm-page-header {
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  flex-wrap: wrap;
  gap: 1rem;
  margin-bottom: 1.5rem;
}

.mm-page-title {
  font-size: 2rem !important;
  font-weight: 800 !important;
  color: var(--mm-ink) !important;
  margin: 0 !important;
}

.mm-page-subtitle {
  color: var(--mm-muted) !important;
  font-size: 0.95rem !important;
  margin: 0.25rem 0 0 !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-write-card-marker),
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-upload-card-marker) {
  background: rgba(255, 255, 255, 0.92) !important;
  border: 1px solid rgba(124, 111, 232, 0.1) !important;
  border-radius: 22px !important;
  padding: 1.5rem 1.6rem !important;
  box-shadow: var(--mm-shadow) !important;
}

.mm-card-spacer {
  height: 0.4rem;
}

.mm-card-header-row {
  display: flex;
  align-items: flex-start;
  gap: 0.9rem;
  margin-bottom: 1.1rem;
  position: relative;
}

.mm-card-icon {
  width: 44px;
  height: 44px;
  min-width: 44px;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 1.3rem;
  border-radius: 12px;
  background: rgba(124, 111, 232, 0.12);
}

.mm-card-icon--teal {
  background: rgba(45, 190, 170, 0.14);
}

.mm-card-title {
  font-size: 1.1rem;
  font-weight: 800;
  color: var(--mm-ink);
}

.mm-card-subtitle {
  font-size: 0.85rem;
  color: var(--mm-muted);
  margin-top: 0.15rem;
}

.mm-journal-illustration,
.mm-upload-illustration {
  position: absolute;
  right: 0;
  top: -0.5rem;
  width: 90px !important;
  max-width: 90px !important;
  height: auto !important;
  max-height: 100px !important;
  opacity: 0.95;
}

.mm-journal-illustration--emoji,
.mm-upload-illustration--emoji {
  font-size: 2.6rem;
  width: auto;
}

@media (max-width: 900px) {
  .mm-journal-illustration, .mm-upload-illustration { display: none; }
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-write-card-marker) [data-testid="stTextArea"] textarea,
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-upload-card-marker) [data-testid="stTextArea"] textarea {
  border-radius: 16px !important;
  border: 1px solid rgba(124, 111, 232, 0.2) !important;
  background: rgba(255, 255, 255, 0.9) !important;
  font-size: 0.95rem !important;
  padding: 0.9rem 1rem !important;
}

.mm-char-counter {
  text-align: right;
  font-size: 0.75rem;
  color: var(--mm-muted);
  margin: 0.3rem 0 0.9rem;
}

.mm-card-hint {
  font-size: 0.78rem;
  color: var(--mm-muted);
  margin: 0.5rem 0 0;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-write-card-marker) div[data-testid="stButton"] button,
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-upload-card-marker) div[data-testid="stButton"] button {
  background: rgba(124, 111, 232, 0.1) !important;
  color: var(--mm-primary-dark) !important;
  border: 1px solid rgba(124, 111, 232, 0.3) !important;
  border-radius: 999px !important;
  font-weight: 700 !important;
  padding: 0.6rem 1.4rem !important;
  box-shadow: none !important;
  width: auto !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-write-card-marker) div[data-testid="stButton"] button p,
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-upload-card-marker) div[data-testid="stButton"] button p {
  color: var(--mm-primary-dark) !important;
  font-weight: 700 !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-upload-card-marker) [data-testid="stFileUploaderDropzone"] {
  border-radius: 16px !important;
  border: 2px dashed rgba(124, 111, 232, 0.3) !important;
  background: rgba(124, 111, 232, 0.03) !important;
}

.mm-section-heading-row {
  display: flex;
  align-items: center;
  gap: 0.5rem;
  margin: 1.5rem 0 0.9rem;
}

.mm-section-heading-icon {
  width: 32px;
  height: 32px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: rgba(124, 111, 232, 0.12);
  font-size: 0.95rem;
}

.mm-section-heading-text {
  font-size: 1.15rem;
  font-weight: 800;
  color: var(--mm-ink);
}

[data-testid="stExpander"] {
  border-radius: 14px !important;
  border: 1px solid rgba(124, 111, 232, 0.12) !important;
  background: rgba(255, 255, 255, 0.85) !important;
  margin-bottom: 0.6rem !important;
  box-shadow: 0 4px 14px rgba(70, 50, 120, 0.05) !important;
}

.mm-page-footer-quote {
  text-align: center;
  font-size: 0.85rem;
  color: var(--mm-muted);
  margin-top: 1.5rem;
  padding: 0.5rem;
}

/* ==========================================================================
   Wellness Chat page
   ========================================================================== */

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-intro-card-marker),
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-chat-area-marker) {
  background: rgba(255, 255, 255, 0.92) !important;
  border: 1px solid rgba(124, 111, 232, 0.1) !important;
  border-radius: 22px !important;
  padding: 1.6rem 1.7rem !important;
  box-shadow: var(--mm-shadow) !important;
  margin-bottom: 1.2rem !important;
}

.mm-chat-intro-row {
  display: flex;
  align-items: center;
  gap: 1.4rem;
  margin-bottom: 1.3rem;
}

.mm-chat-mascot--emoji {
  font-size: 4rem;
}

.mm-chat-intro-title {
  font-size: 1.3rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-bottom: 0.4rem;
}

.mm-chat-intro-desc {
  font-size: 0.92rem;
  color: var(--mm-muted);
  line-height: 1.55;
}

.mm-trust-badge-row {
  display: flex;
  gap: 0.9rem;
  flex-wrap: wrap;
}

.mm-trust-badge {
  flex: 1;
  min-width: 180px;
  display: flex;
  align-items: flex-start;
  gap: 0.7rem;
  background: rgba(124, 111, 232, 0.06);
  border-radius: 14px;
  padding: 0.9rem 1rem;
}

.mm-trust-badge-icon {
  width: 36px;
  height: 36px;
  min-width: 36px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  background: rgba(124, 111, 232, 0.14);
  font-size: 1rem;
}

.mm-trust-badge-title {
  font-size: 0.88rem;
  font-weight: 700;
  color: var(--mm-ink);
}

.mm-trust-badge-desc {
  font-size: 0.78rem;
  color: var(--mm-muted);
  line-height: 1.4;
}

.mm-chat-empty-state {
  text-align: center;
  padding: 2.5rem 1rem;
}

.mm-chat-empty-illustration--emoji {
  font-size: 4rem;
}

.mm-chat-empty-title {
  font-size: 1.3rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-top: 1rem;
}

.mm-chat-empty-desc {
  font-size: 0.92rem;
  color: var(--mm-muted);
  margin-top: 0.4rem;
  line-height: 1.55;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-chat-area-marker) div[data-testid="stButton"] button {
  background: rgba(124, 111, 232, 0.08) !important;
  color: var(--mm-primary-dark) !important;
  border: 1px solid rgba(124, 111, 232, 0.25) !important;
  border-radius: 999px !important;
  font-size: 0.8rem !important;
  font-weight: 600 !important;
  box-shadow: none !important;
}

[data-testid="stChatInput"] textarea,
[data-testid="stChatInputTextArea"] {
  border-radius: 999px !important;
  border: 1px solid rgba(124, 111, 232, 0.2) !important;
  background: rgba(255, 255, 255, 0.95) !important;
}

[data-testid="stChatInput"] button {
  background: var(--mm-primary) !important;
  border-radius: 50% !important;
}

[data-testid="stChatMessage"] {
  background: rgba(124, 111, 232, 0.05) !important;
  border-radius: 14px !important;
  padding: 0.6rem 0.9rem !important;
}

/* ==========================================================================
   Face Detection page
   ========================================================================== */

.mm-scan-label {
  font-size: 0.95rem;
  font-weight: 700;
  color: var(--mm-ink);
  margin: 0.5rem 0 0.8rem;
}

/* Style native Streamlit tabs to look like the reference's underlined tabs */
[data-testid="stTabs"] [data-baseweb="tab-list"] {
  gap: 1.5rem;
  border-bottom: 1px solid rgba(124, 111, 232, 0.15);
}

[data-testid="stTabs"] button[data-baseweb="tab"] {
  background: transparent !important;
  font-weight: 700 !important;
  color: var(--mm-muted) !important;
  padding: 0.6rem 0.2rem !important;
}

[data-testid="stTabs"] button[aria-selected="true"] {
  color: var(--mm-primary) !important;
  border-bottom: 3px solid var(--mm-primary) !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-scan-card-marker) {
  background: rgba(255, 255, 255, 0.9) !important;
  border: 2px dashed rgba(124, 111, 232, 0.25) !important;
  border-radius: 22px !important;
  padding: 2rem !important;
}

.mm-scan-placeholder {
  text-align: center;
  padding: 1.5rem 0 1rem;
}

.mm-scan-frame-icon {
  width: 120px;
  height: 120px;
  margin: 0 auto 1.2rem;
  display: flex;
  align-items: center;
  justify-content: center;
}

.mm-scan-frame-icon--emoji {
  font-size: 3rem;
  border-radius: 50%;
  background: rgba(124, 111, 232, 0.1);
  border: 2px solid rgba(124, 111, 232, 0.3);
}

.mm-scan-placeholder-title {
  font-size: 1.3rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-bottom: 0.5rem;
}

.mm-scan-placeholder-desc {
  font-size: 0.9rem;
  color: var(--mm-muted);
  line-height: 1.55;
  margin-bottom: 1.4rem;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-scan-card-marker) div[data-testid="stButton"] button,
div[data-testid="stVerticalBlock"]:has(.mm-start-camera-marker) div[data-testid="stButton"] button,
div[data-testid="stVerticalBlock"]:has(.mm-start-camera-marker) button {
  background: linear-gradient(100deg, var(--mm-primary-dark) 0%, var(--mm-primary) 100%) !important;
  color: #ffffff !important;
  border: none !important;
  border-radius: 999px !important;
  font-weight: 700 !important;
  font-size: 1rem !important;
  padding: 0.85rem 1.6rem !important;
  box-shadow: 0 12px 30px rgba(91, 79, 196, 0.4) !important;
  transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-scan-card-marker) div[data-testid="stButton"] button p,
div[data-testid="stVerticalBlock"]:has(.mm-start-camera-marker) div[data-testid="stButton"] button p,
div[data-testid="stVerticalBlock"]:has(.mm-start-camera-marker) button p {
  color: #ffffff !important;
  font-weight: 700 !important;
  font-size: 1rem !important;
}

div[data-testid="stVerticalBlock"]:has(.mm-start-camera-marker) button:hover {
  transform: translateY(-2px);
  box-shadow: 0 16px 38px rgba(91, 79, 196, 0.5) !important;
}

.mm-privacy-note {
  background: rgba(124, 111, 232, 0.08);
  border-radius: 14px;
  padding: 0.9rem 1.1rem;
  margin-top: 1.5rem;
  font-size: 0.88rem;
  color: var(--mm-ink);
  text-align: center;
}

.mm-privacy-note span {
  color: var(--mm-muted);
  font-weight: 400;
}

.mm-tips-footer {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 1rem;
  background: rgba(255, 255, 255, 0.85);
  border: 1px solid rgba(124, 111, 232, 0.1);
  border-radius: 18px;
  padding: 1.2rem 1.5rem;
  margin-top: 1.2rem;
  box-shadow: var(--mm-shadow);
}

.mm-tips-title {
  font-size: 0.95rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-bottom: 0.3rem;
}

.mm-tips-desc {
  font-size: 0.85rem;
  color: var(--mm-muted);
}

.mm-tips-illustration--emoji {
  font-size: 2.2rem;
}

/* ==========================================================================
   Relax page
   ========================================================================== */

.mm-relax-top-row {
  display: flex;
  gap: 1.2rem;
  margin-bottom: 0.6rem;
  flex-wrap: wrap;
}

.mm-relax-card {
  flex: 1;
  min-width: 220px;
  border-radius: 20px;
  padding: 1.4rem 1.5rem;
  box-shadow: var(--mm-shadow);
}

.mm-relax-card--purple { background: linear-gradient(160deg, rgba(124,111,232,0.14) 0%, rgba(124,111,232,0.06) 100%); }
.mm-relax-card--blue   { background: linear-gradient(160deg, rgba(59,130,246,0.14) 0%, rgba(59,130,246,0.06) 100%); }
.mm-relax-card--green  { background: linear-gradient(160deg, rgba(34,197,94,0.14) 0%, rgba(34,197,94,0.06) 100%); }

.mm-relax-card-title {
  font-size: 1.1rem;
  font-weight: 800;
  color: var(--mm-ink);
  margin-bottom: 0.4rem;
}

.mm-relax-card--purple .mm-relax-card-title { color: var(--mm-primary-dark); }
.mm-relax-card--blue .mm-relax-card-title { color: #1d4ed8; }
.mm-relax-card--green .mm-relax-card-title { color: #15803d; }

.mm-relax-card-desc {
  font-size: 0.85rem;
  color: var(--mm-muted);
  margin-bottom: 1rem;
  line-height: 1.5;
}

.mm-relax-card-btn {
  display: inline-block;
  background: linear-gradient(100deg, var(--mm-primary-dark) 0%, var(--mm-primary) 100%);
  color: #ffffff !important;
  font-weight: 700;
  font-size: 0.85rem;
  padding: 0.55rem 1.1rem;
  border-radius: 999px;
  text-decoration: none !important;
  box-shadow: 0 8px 20px rgba(91, 79, 196, 0.3);
}

/* Real music-selection buttons (rendered below the cards via st.button),
   colored to match their respective card themes above them. */
div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--blue) button {
  background: linear-gradient(100deg, #1d4ed8 0%, #3b82f6 100%) !important;
  color: #ffffff !important;
  border: none !important;
  border-radius: 999px !important;
  font-weight: 700 !important;
  padding: 0.7rem 1.3rem !important;
  box-shadow: 0 10px 24px rgba(59, 130, 246, 0.35) !important;
}

div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--green) button {
  background: linear-gradient(100deg, #15803d 0%, #22c55e 100%) !important;
  color: #ffffff !important;
  border: none !important;
  border-radius: 999px !important;
  font-weight: 700 !important;
  padding: 0.7rem 1.3rem !important;
  box-shadow: 0 10px 24px rgba(34, 197, 94, 0.35) !important;
}

div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--blue) button p,
div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--green) button p {
  color: #ffffff !important;
  font-weight: 700 !important;
}

div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--blue) button:hover,
div[data-testid="stVerticalBlock"]:has(.mm-relax-btn-marker--green) button:hover {
  transform: translateY(-2px);
}

.mm-section-heading-row#mm-music-section,
.mm-section-heading-row {
  margin: 0.8rem 0 0.9rem;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-music-card-marker),
[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-breathing-card-marker) {
  background: rgba(255, 255, 255, 0.9) !important;
  border: 1px solid rgba(124, 111, 232, 0.1) !important;
  border-radius: 22px !important;
  padding: 1.4rem !important;
  box-shadow: var(--mm-shadow) !important;
}

.mm-breathing-row {
  display: flex;
  align-items: flex-start;
  gap: 1rem;
  justify-content: space-between;
}

.mm-breathing-illustration--emoji { font-size: 3.5rem; }

.breathing-container {
  display: flex;
  align-items: center;
  justify-content: center;
  margin: 1.5rem 0;
}

.circle {
  width: 130px;
  height: 130px;
  border-radius: 50%;
  background: radial-gradient(circle, rgba(124,111,232,0.35) 0%, rgba(124,111,232,0.12) 70%);
  border: 2px solid rgba(124, 111, 232, 0.4);
  display: flex;
  align-items: center;
  justify-content: center;
  font-weight: 700;
  color: var(--mm-primary-dark);
  font-size: 0.95rem;
  animation: mm-breathe 6s ease-in-out infinite;
}

@keyframes mm-breathe {
  0%, 100% { transform: scale(0.85); }
  50% { transform: scale(1.15); }
}

.mm-relax-quote-card {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 1rem;
  background: rgba(124, 111, 232, 0.08);
  border-radius: 20px;
  padding: 1.4rem 1.7rem;
  margin-top: 1.4rem;
}

.mm-relax-quote-text {
  font-size: 1rem;
  font-style: italic;
  color: var(--mm-ink);
  line-height: 1.55;
  max-width: 640px;
}

.mm-relax-quote-illustration--emoji { font-size: 2.5rem; }

/* ==========================================================================
   Dashboard page
   ========================================================================== */

.mm-dash-metric {
  background: rgba(255, 255, 255, 0.9);
  border: 1px solid rgba(124, 111, 232, 0.1);
  border-top: 3px solid var(--accent, var(--mm-primary));
  border-radius: 18px;
  padding: 1.1rem 1.2rem;
  box-shadow: var(--mm-shadow);
  display: flex;
  align-items: center;
  gap: 0.8rem;
}

.mm-dash-metric-icon {
  width: 48px;
  height: 48px;
  min-width: 48px;
  display: flex;
  align-items: center;
  justify-content: center;
  border-radius: 50%;
  font-size: 1.4rem;
  background: color-mix(in srgb, var(--accent, var(--mm-primary)) 16%, white);
}

.mm-dash-metric-label {
  font-size: 0.78rem;
  color: var(--mm-muted);
  font-weight: 600;
}

.mm-dash-metric-value {
  font-size: 1.35rem;
  font-weight: 800;
  color: var(--mm-ink);
  line-height: 1.2;
}

.mm-dash-metric-sub {
  font-size: 0.72rem;
  color: var(--accent, var(--mm-primary));
  font-weight: 600;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-dash-card-marker) {
  background: rgba(255, 255, 255, 0.92) !important;
  border: 1px solid rgba(124, 111, 232, 0.1) !important;
  border-radius: 20px !important;
  padding: 1.3rem 1.4rem !important;
  box-shadow: var(--mm-shadow) !important;
  height: 100%;
}

/* Donut */
.mm-donut-row {
  display: flex;
  align-items: center;
  gap: 1.2rem;
  margin-top: 1rem;
}

.mm-donut-ring {
  width: 130px;
  height: 130px;
  min-width: 130px;
  border-radius: 50%;
  position: relative;
}

.mm-donut-ring::after {
  content: "";
  position: absolute;
  inset: 22px;
  background: #ffffff;
  border-radius: 50%;
}

.mm-donut-legend {
  display: flex;
  flex-direction: column;
  gap: 0.5rem;
}

.mm-donut-legend-item {
  font-size: 0.85rem;
  color: var(--mm-ink);
  display: flex;
  align-items: center;
  gap: 0.5rem;
}

.mm-donut-legend-dot {
  width: 10px;
  height: 10px;
  border-radius: 50%;
  display: inline-block;
}

/* Trend line */
.mm-trend-svg {
  width: 100%;
  height: 140px;
  margin-top: 0.8rem;
}

.mm-trend-labels {
  display: flex;
  justify-content: space-between;
  font-size: 0.68rem;
  color: var(--mm-muted);
  padding: 0 10px;
}

/* Emotion bars */
.mm-bar-chart {
  display: flex;
  align-items: flex-end;
  justify-content: space-around;
  gap: 0.8rem;
  height: 150px;
  margin-top: 1rem;
}

.mm-bar-col {
  display: flex;
  flex-direction: column;
  align-items: center;
  gap: 0.3rem;
  flex: 1;
}

.mm-bar-track {
  width: 32px;
  height: 100px;
  display: flex;
  align-items: flex-end;
}

.mm-bar-fill {
  width: 100%;
  background: linear-gradient(180deg, var(--mm-primary) 0%, var(--mm-primary-dark) 100%);
  border-radius: 8px 8px 0 0;
  min-height: 4px;
}

.mm-bar-label {
  font-size: 0.75rem;
  color: var(--mm-ink);
}

.mm-bar-value {
  font-size: 0.7rem;
  color: var(--mm-muted);
}

/* Activity table */
.mm-activity-table {
  width: 100%;
  border-collapse: collapse;
  margin-top: 0.8rem;
  font-size: 0.85rem;
}

.mm-activity-table th {
  text-align: left;
  color: var(--mm-muted);
  font-weight: 700;
  font-size: 0.72rem;
  text-transform: uppercase;
  padding: 0.5rem 0.6rem;
  border-bottom: 1px solid rgba(124, 111, 232, 0.12);
}

.mm-activity-table td {
  padding: 0.6rem;
  border-bottom: 1px solid rgba(124, 111, 232, 0.06);
  color: var(--mm-ink);
}

.mm-view-all-label {
  text-align: center;
  margin-top: 0.9rem;
  font-size: 0.85rem;
  font-weight: 700;
  color: var(--mm-primary-dark);
  opacity: 0.7;
}

/* Weekly insights + export */
.mm-insight-row {
  display: flex;
  align-items: center;
  justify-content: space-between;
  gap: 1rem;
}

.mm-insight-text {
  font-size: 0.85rem;
  color: var(--mm-muted);
  line-height: 1.55;
  margin-top: 0.5rem;
}

.mm-insight-illustration--emoji { font-size: 2.5rem; }

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-export-card-marker) div[data-testid="stButton"] button,
div[data-testid="stVerticalBlock"]:has(.mm-export-btn-marker) button {
  background: linear-gradient(100deg, var(--mm-primary-dark) 0%, var(--mm-primary) 100%) !important;
  color: #ffffff !important;
  border: none !important;
  border-radius: 999px !important;
  font-weight: 700 !important;
  font-size: 0.95rem !important;
  padding: 0.7rem 1.4rem !important;
  box-shadow: 0 10px 26px rgba(91, 79, 196, 0.4) !important;
  transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}

[data-testid="stVerticalBlockBorderWrapper"]:has(.mm-export-card-marker) div[data-testid="stButton"] button p,
div[data-testid="stVerticalBlock"]:has(.mm-export-btn-marker) button p {
  color: #ffffff !important;
  font-weight: 700 !important;
}

div[data-testid="stVerticalBlock"]:has(.mm-export-btn-marker) button:hover {
  transform: translateY(-2px);
  box-shadow: 0 14px 32px rgba(91, 79, 196, 0.5) !important;
}

/* ==========================================================================
   Make sidebar permanently visible / non-collapsible
   ========================================================================== */

/* Force the sidebar to stay open regardless of Streamlit's internal
   collapsed state (aria-expanded toggles when the user clicks collapse). */
section[data-testid="stSidebar"],
section[data-testid="stSidebar"][aria-expanded="false"],
section[data-testid="stSidebar"][aria-expanded="true"] {
  min-width: 260px !important;
  max-width: 260px !important;
  width: 260px !important;
  transform: none !important;
  visibility: visible !important;
  margin-left: 0 !important;
}

/* Hide every version of the collapse/expand toggle control across
   Streamlit versions, so it can never be triggered again. */
button[data-testid="stSidebarCollapseButton"],
div[data-testid="stSidebarCollapsedControl"],
[data-testid="collapsedControl"],
button[aria-label="Close sidebar"],
button[aria-label="Open sidebar"],
button[title="Close sidebar"],
button[title="Open sidebar"] {
  display: none !important;
  visibility: hidden !important;
  pointer-events: none !important;
}

/* Ensure main content area doesn't shift to fill space if a collapsed
   state briefly applies mid-render. */
div[data-testid="stAppViewContainer"] > .main {
  margin-left: 0 !important;
}

Overwriting styles/dashboard.css


In [ ]:
%%writefile components/journal.py
"""
components/journal.py

Premium "Journal" section for MoodMentor, matching the reference design:
a "write your entry" card with AI mood analysis, a file-upload card, and a
styled past-entries list.

IMPORTANT: UI ONLY. Every backend call (POST /analyze-text, POST /analyze),
every db call (save_mood_log, get_user_mood_history), and every condition
is copied verbatim from the original inline "Journal" branch in app.py.
Only the layout/markup changed.

One small, disclosed UX choice: the reference screenshot shows no visible
"confirm" button for the file-upload flow (just a dropzone). The original
code required an explicit "Run NLP Analysis on file" click after choosing
a file. I kept that explicit confirm step rather than silently changing
behavior to auto-analyze on upload -- flagging this in case you'd prefer
the auto-trigger version instead.

Usage in app.py (replaces the original `elif section == "Journal": ...` body):

    from components.journal import render_journal_section
    ...
    elif section == "Journal":
        render_journal_section(user, st.session_state.token)
"""

import base64
import os
from datetime import datetime
from pathlib import Path

import requests
import streamlit as st

from db import MOOD_EMOJI, get_user_mood_history, save_mood_log

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#2ecc71"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#3498db"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#e67e22"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}


def _style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})


_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _img_or_emoji_html(rel_path: tuple, emoji_fallback: str, css_class: str, width_px: int = 90) -> str:
    """Use a generated illustration if present, else fall back to an emoji.

    width_px is applied as both an inline attribute AND via CSS class, so
    the image is capped at a sane size even if the stylesheet fails to
    load for any reason -- it won't render at native (huge) resolution.
    """
    path = _resolve_first(rel_path)
    b64 = _get_base64_of_file(str(path))
    if b64:
        return (
            f'<img src="data:image/png;base64,{b64}" class="{css_class}" '
            f'width="{width_px}" style="width:{width_px}px;height:auto;" alt="" />'
        )
    return f'<div class="{css_class} {css_class}--emoji">{emoji_fallback}</div>'


def _render_page_header() -> None:
    now = datetime.now()
    st.markdown(
        f"""
        <div class="mm-page-header">
            <div>
                <h1 class="mm-page-title">Journal</h1>
                <p class="mm-page-subtitle">A safe space to reflect and express yourself 💜</p>
            </div>
            <div class="mm-home-datetime">
                📅 {now.strftime('%A, %-d %B %Y')} &nbsp;|&nbsp; 🕐 {now.strftime('%I:%M %p')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_analysis_result(r: dict) -> None:
    confidence = r.get("emotion_confidence")
    conf_str = f", Confidence: **{confidence:.0%}**" if confidence is not None else ""
    st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
               f"Emotion: **{r['final_emotion']}**{conf_str}")
    st.bar_chart(r["emotion_scores"])
    if r.get("recommendation"):
        st.info(f"**Recommendation:** {r['recommendation']}")


def render_journal_section(user: dict, token: str, debug: bool = True) -> None:
    """Render the Journal section. Identical backend calls/db calls to the
    original inline `elif section == "Journal":` block -- only the
    layout/markup is new.

    `debug=True` (default) shows a warning if dashboard.css can't be found,
    since this page's card/illustration styling depends entirely on it
    (loaded once via render_sidebar(), which runs before this on every
    authenticated page).
    """

    if debug:
        css_path = _resolve_first(
            ("styles", "dashboard.css"),
            ("assets", "styles", "dashboard.css"),
            ("assets", "css", "dashboard.css"),
        )
        if not css_path.exists():
            st.warning(
                "dashboard.css not found -- Journal page will look "
                f"unstyled. Checked: `{css_path}` (and sibling candidate roots)."
            )

    headers = {"Authorization": f"Bearer {token}"}

    _render_page_header()

    # -------------------------------------------------------------- #
    # Write & analyze card
    # -------------------------------------------------------------- #
    book_html = _img_or_emoji_html(
        ("assets", "illustrations", "journal-book.png"), "📔", "mm-journal-illustration"
    )
    with st.container(border=True):
        st.markdown('<div class="mm-write-card-marker"></div>', unsafe_allow_html=True)
        st.markdown(
            f"""
            <div class="mm-card-header-row">
                <div class="mm-card-icon">📝</div>
                <div>
                    <div class="mm-card-title">Write about how you're feeling today</div>
                    <div class="mm-card-subtitle">Let your thoughts flow freely. This is your space.</div>
                </div>
                {book_html}
            </div>
            """,
            unsafe_allow_html=True,
        )

        journal_text = st.text_area(
            "Write about how you're feeling today", height=150,
            placeholder="Your note here...", label_visibility="collapsed",
            max_chars=2000, key="journal_text_input",
        )
        st.markdown(
            f'<div class="mm-char-counter">{len(journal_text)} / 2000</div>',
            unsafe_allow_html=True,
        )

        analyze_clicked = st.button("✨  Analyze my mood", key="journal_analyze_btn")
        st.markdown(
            '<p class="mm-card-hint">Get AI insights about your current mood</p>',
            unsafe_allow_html=True,
        )

        if analyze_clicked:
            if not journal_text.strip():
                st.warning("Write something first.")
            else:
                with st.spinner("Running NLP analysis…"):
                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/analyze-text",
                            json={"text": journal_text},
                            headers=headers, timeout=120,
                        )
                    except requests.exceptions.RequestException as e:
                        st.error(f"Could not reach backend: {e}")
                        resp = None
                if resp is not None:
                    if resp.status_code != 200:
                        st.error("Analysis failed.")
                    else:
                        r = resp.json()
                        save_mood_log(
                            user["id"], r["final_sentiment"], r["final_emotion"],
                            r["sentiment_scores"]["compound"], journal_text,
                            confidence=r.get("emotion_confidence"),
                        )
                        _render_analysis_result(r)

    st.markdown('<div class="mm-card-spacer"></div>', unsafe_allow_html=True)

    # -------------------------------------------------------------- #
    # File upload card
    # -------------------------------------------------------------- #
    upload_html = _img_or_emoji_html(
        ("assets", "illustrations", "upload-cloud.png"), "☁️", "mm-upload-illustration"
    )
    with st.container(border=True):
        st.markdown('<div class="mm-upload-card-marker"></div>', unsafe_allow_html=True)
        st.markdown(
            f"""
            <div class="mm-card-header-row">
                <div class="mm-card-icon mm-card-icon--teal">📤</div>
                <div>
                    <div class="mm-card-title">Or upload a file</div>
                    <div class="mm-card-subtitle">Upload a CSV or TXT file to analyze your mood data.</div>
                </div>
                {upload_html}
            </div>
            """,
            unsafe_allow_html=True,
        )

        uploaded = st.file_uploader(
            "Choose a CSV or TXT file", type=["csv", "txt"], label_visibility="collapsed",
        )
        st.markdown(
            '<p class="mm-card-hint">CSV, TXT files only &bull; Max size 200MB</p>',
            unsafe_allow_html=True,
        )

        if uploaded is not None:
            run_clicked = st.button("Run NLP Analysis on file", key="journal_file_analyze_btn")
            if run_clicked:
                files = {"file": (uploaded.name, uploaded.getvalue())}
                with st.spinner("Running multilingual NLP pipeline…"):
                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/analyze", files=files, headers=headers, timeout=120,
                        )
                    except requests.exceptions.RequestException as e:
                        st.error(f"Could not reach backend: {e}")
                        resp = None
                if resp is not None:
                    if resp.status_code != 200:
                        st.error("Analysis failed.")
                    else:
                        r = resp.json()
                        save_mood_log(
                            user["id"], r["final_sentiment"], r["final_emotion"],
                            r["sentiment_scores"]["compound"], r.get("cleaned_text", ""),
                            confidence=r.get("emotion_confidence"),
                        )
                        _render_analysis_result(r)

    st.markdown('<div class="mm-card-spacer"></div>', unsafe_allow_html=True)

    # -------------------------------------------------------------- #
    # Past entries
    # -------------------------------------------------------------- #
    st.markdown(
        """
        <div class="mm-section-heading-row">
            <span class="mm-section-heading-icon">🕐</span>
            <span class="mm-section-heading-text">Past entries</span>
        </div>
        """,
        unsafe_allow_html=True,
    )

    history = [h for h in get_user_mood_history(user["id"], limit=20) if h["journal_text"]]
    if not history:
        st.caption("No journal entries yet.")
    for h in history:
        s = _style_for(h["sentiment"])
        conf_str = f" · Confidence: {h['confidence']:.0%}" if h.get("confidence") is not None else ""
        with st.expander(
            f"{s['emoji']}  {h['sentiment']} — {h['created_at'].strftime('%Y-%m-%d %H:%M')}{conf_str}"
        ):
            st.write(h["journal_text"])

    st.markdown(
        """
        <div class="mm-page-footer-quote">
            🍃 Small steps every day lead to a better you. Keep going! 💜 🍃
        </div>
        """,
        unsafe_allow_html=True,
    )


Overwriting components/journal.py


In [ ]:
%%writefile components/wellness_chat.py
"""
components/wellness_chat.py

Premium "Wellness Chat" section for MoodMentor, matching the reference
design: an intro card with trust badges, an empty/active chat area, and a
styled chat input bar.

IMPORTANT: UI ONLY. Every backend call (POST /chat), every session_state
key (chat_history), and every condition is copied verbatim from the
original inline "Wellness Chat" branch in app.py. Only the layout/markup
changed.

Deliberately NOT added: a functional paperclip/attachment button. The
reference screenshot shows one, but there's no file-attachment backend
for chat, so a clickable paperclip would imply functionality that doesn't
exist. Left out rather than faked.

Usage in app.py (replaces the original `elif section == "Wellness Chat": ...` body):

    from components.wellness_chat import render_wellness_chat_section
    ...
    elif section == "Wellness Chat":
        render_wellness_chat_section(st.session_state.token)
"""

import base64
import os
from datetime import datetime
from pathlib import Path

import requests
import streamlit as st

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _img_or_emoji_html(rel_path: tuple, emoji_fallback: str, css_class: str, width_px: int) -> str:
    path = _resolve_first(rel_path)
    b64 = _get_base64_of_file(str(path))
    if b64:
        return (
            f'<img src="data:image/png;base64,{b64}" class="{css_class}" '
            f'width="{width_px}" style="width:{width_px}px;height:auto;" alt="" />'
        )
    return f'<div class="{css_class} {css_class}--emoji">{emoji_fallback}</div>'


TRUST_BADGES = [
    {"icon": "🛡️", "title": "Private & Safe", "desc": "Your conversations are confidential"},
    {"icon": "💗", "title": "Non-Judgmental", "desc": "Talk freely without fear of judgment"},
    {"icon": "🌱", "title": "Here to Help", "desc": "I'm here whenever you need to talk"},
]


def _render_page_header() -> None:
    now = datetime.now()
    st.markdown(
        f"""
        <div class="mm-page-header">
            <div>
                <h1 class="mm-page-title">Wellness Chat 💜</h1>
                <p class="mm-page-subtitle">
                    A supportive space to talk about how you're feeling.<br>
                    Not a substitute for professional care.
                </p>
            </div>
            <div class="mm-home-datetime">
                📅 {now.strftime('%A, %-d %B %Y')} &nbsp;|&nbsp; 🕐 {now.strftime('%I:%M %p')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_intro_card() -> None:
    mascot_html = _img_or_emoji_html(
        ("assets", "illustrations", "mentor-brain-mascot.png"), "🧠", "mm-chat-mascot", 140
    )
    badges_html = "".join(
        f"""
        <div class="mm-trust-badge">
            <div class="mm-trust-badge-icon">{b['icon']}</div>
            <div>
                <div class="mm-trust-badge-title">{b['title']}</div>
                <div class="mm-trust-badge-desc">{b['desc']}</div>
            </div>
        </div>
        """
        for b in TRUST_BADGES
    )
    with st.container(border=True):
        st.markdown('<div class="mm-intro-card-marker"></div>', unsafe_allow_html=True)
        st.markdown(
            f"""
            <div class="mm-chat-intro-row">
                {mascot_html}
                <div>
                    <div class="mm-chat-intro-title">Hi there! I'm MoodMentor 👋</div>
                    <div class="mm-chat-intro-desc">
                        I'm here to listen and support you.<br>
                        Share what's on your mind &mdash; big or small.
                    </div>
                </div>
            </div>
            <div class="mm-trust-badge-row">{badges_html}</div>
            """,
            unsafe_allow_html=True,
        )


def _render_empty_state() -> None:
    bubbles_html = _img_or_emoji_html(
        ("assets", "illustrations", "chat-bubbles.png"), "💬", "mm-chat-empty-illustration", 160
    )
    st.markdown(
        f"""
        <div class="mm-chat-empty-state">
            {bubbles_html}
            <div class="mm-chat-empty-title">Let's start a conversation</div>
            <div class="mm-chat-empty-desc">
                Share how you're feeling today.<br>I'm here to listen.
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_wellness_chat_section(token: str, debug: bool = True) -> None:
    """Render the Wellness Chat section. Identical backend calls/session
    state to the original inline `elif section == "Wellness Chat":` block
    -- only the layout/markup is new.

    `debug=True` (default) shows a warning if dashboard.css can't be found.
    """
    if debug:
        css_path = _resolve_first(
            ("styles", "dashboard.css"),
            ("assets", "styles", "dashboard.css"),
            ("assets", "css", "dashboard.css"),
        )
        if not css_path.exists():
            st.warning(
                "dashboard.css not found -- Wellness Chat page will look "
                f"unstyled. Checked: `{css_path}` (and sibling candidate roots)."
            )

    headers = {"Authorization": f"Bearer {token}"}

    _render_page_header()
    _render_intro_card()

    with st.container(border=True):
        st.markdown('<div class="mm-chat-area-marker"></div>', unsafe_allow_html=True)

        if not st.session_state.chat_history:
            _render_empty_state()
        else:
            top_l, top_r = st.columns([5, 1])
            with top_r:
                if st.button("Clear chat", key="wellness_clear_chat_btn"):
                    st.session_state.chat_history = []
                    st.rerun()

            chat_box = st.container(height=380)
            with chat_box:
                for turn in st.session_state.chat_history:
                    with st.chat_message(turn["role"]):
                        st.write(turn["content"])

    user_msg = st.chat_input("How are you feeling today?")
    if user_msg:
        st.session_state.chat_history.append({"role": "user", "content": user_msg})
        recent_history = st.session_state.chat_history[-10:-1]
        try:
            resp = requests.post(
                f"{BACKEND_URL}/chat",
                json={"message": user_msg, "history": recent_history},
                headers=headers, timeout=60,
            )
            reply = resp.json()["reply"] if resp.status_code == 200 else \
                "Sorry, I couldn't reach the wellness assistant right now."
        except requests.exceptions.RequestException:
            reply = "Sorry, I couldn't reach the wellness assistant right now."
        st.session_state.chat_history.append({"role": "assistant", "content": reply})
        st.rerun()

Overwriting components/wellness_chat.py


In [ ]:
%%writefile components/face_detection.py
"""
components/face_detection.py

Premium "Face Scan & Recommendations" section for MoodMentor.

IMPORTANT: UI ONLY. The `analyze_and_display()` function -- every cv2/numpy/
DeepFace call, every mood-based advice branch, save_face_scan() call, and
error handling -- is copied verbatim from the original inline
"Face Detection" branch in app.py. Only the layout/markup around it changed.

One disclosed, non-backend UX change: the reference shows a "Start Camera"
button that reveals the camera widget, rather than the camera opening
immediately on page load (which is what st.camera_input does natively).
This is gated with a plain session_state flag -- analyze_and_display()
itself is untouched.

Usage in app.py (replaces the original `elif section == "Face Detection": ...` body):

    from components.face_detection import render_face_detection_section
    ...
    elif section == "Face Detection":
        render_face_detection_section(user)
"""

import base64
import os
from datetime import datetime
from pathlib import Path

import streamlit as st

from db import save_face_scan

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _img_or_emoji_html(rel_path: tuple, emoji_fallback: str, css_class: str, width_px: int) -> str:
    path = _resolve_first(rel_path)
    b64 = _get_base64_of_file(str(path))
    if b64:
        return (
            f'<img src="data:image/png;base64,{b64}" class="{css_class}" '
            f'width="{width_px}" style="width:{width_px}px;height:auto;" alt="" />'
        )
    return f'<div class="{css_class} {css_class}--emoji">{emoji_fallback}</div>'


def _render_page_header() -> None:
    now = datetime.now()
    st.markdown(
        f"""
        <div class="mm-page-header">
            <div>
                <h1 class="mm-page-title">📸 Face Scan &amp; Recommendations</h1>
                <p class="mm-page-subtitle">
                    Using DeepFace AI to read your micro-expressions and provide personalized mentorship.
                </p>
            </div>
            <div class="mm-home-datetime">
                📅 {now.strftime('%A, %-d %B %Y')} &nbsp;|&nbsp; 🕐 {now.strftime('%I:%M %p')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _analyze_and_display(user: dict, image_bytes: bytes) -> None:
    """UNCHANGED from the original inline function -- same cv2/DeepFace
    calls, same save_face_scan() call, same advice branches, same error
    handling. Only moved into its own function scope."""
    import cv2
    import numpy as np
    from deepface import DeepFace

    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        st.info("Scanning biometric markers with MTCNN...")

        tmp_path = "temp_scan.jpg"
        cv2.imwrite(tmp_path, img)

        results = DeepFace.analyze(
            img_path=tmp_path, actions=['emotion'], enforce_detection=True, detector_backend='mtcnn'
        )
        if not isinstance(results, list):
            results = [results]

        st.success(f"Biometric Scan Complete! {len(results)} face(s) mapped.")

        for i, face_data in enumerate(results):
            emotion = face_data['dominant_emotion']
            score_val = face_data['emotion'][emotion]
            box = face_data['region']
            x, y, w, h = box['x'], box['y'], box['w'], box['h']

            cv2.rectangle(img_rgb, (x, y), (x + w, y + h), (0, 255, 120), 3)
            cv2.circle(img_rgb, (x, y), 5, (255, 255, 255), -1)
            cv2.circle(img_rgb, (x + w, y), 5, (255, 255, 255), -1)
            cv2.circle(img_rgb, (x, y + h), 5, (255, 255, 255), -1)
            cv2.circle(img_rgb, (x + w, y + h), 5, (255, 255, 255), -1)
            cv2.putText(
                img_rgb, f"{emotion.upper()} {score_val:.1f}%", (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 120), 2,
            )

            save_face_scan(user["id"], emotion, float(score_val) / 100.0)

        st.image(img_rgb, channels="RGB", use_container_width=True)

        st.markdown("---")
        st.markdown("<h3>🧠 Mood Mentor Analysis</h3>", unsafe_allow_html=True)
        emotion = results[0]['dominant_emotion'].lower()
        if emotion in ["happy", "joy", "amazing"]:
            st.info("💡 **Mentor's Advice:** You're radiating positive energy! Channel this into your "
                    "most challenging tasks today, or share your good mood by helping a colleague.")
            st.balloons()
        elif emotion in ["sad", "sadness"]:
            st.warning("💡 **Mentor's Advice:** It's okay to feel down. Please take a moment for yourself. "
                       "Try the 'Relax' tab for some guided breathing, or write your thoughts down in the Journal.")
        elif emotion in ["angry", "anger", "disgust"]:
            st.error("💡 **Mentor's Advice:** You seem frustrated. Step away from your screen for 5 minutes, "
                     "get a glass of water, and try the 4-7-8 breathing technique in the Relax tab.")
        elif emotion in ["fear", "surprise"]:
            st.warning("💡 **Mentor's Advice:** Take a deep breath. Focus on what you can control right now. "
                       "If you're feeling overwhelmed, break your tasks into smaller steps.")
        else:
            st.success("💡 **Mentor's Advice:** You seem balanced and focused. It's a great time to tackle "
                       "deep work and maintain this calm state.")

    except ValueError:
        st.error("No face detected. Please ensure your face is clearly visible and try again.")
    except Exception as e:
        st.error(f"Error during scan: {e}")
    finally:
        if os.path.exists("temp_scan.jpg"):
            os.remove("temp_scan.jpg")


def _render_camera_tab(user: dict) -> None:
    st.markdown('<p class="mm-scan-label">Initiate Biometric Camera Scan</p>', unsafe_allow_html=True)

    if "face_scan_camera_started" not in st.session_state:
        st.session_state.face_scan_camera_started = False

    with st.container(border=True):
        st.markdown('<div class="mm-scan-card-marker"></div>', unsafe_allow_html=True)

        if not st.session_state.face_scan_camera_started:
            frame_icon_html = _img_or_emoji_html(
                ("assets", "illustrations", "face-scan-icon.png"), "🙂", "mm-scan-frame-icon", 120
            )
            st.markdown(
                f"""
                <div class="mm-scan-placeholder">
                    {frame_icon_html}
                    <div class="mm-scan-placeholder-title">Ready to scan your face</div>
                    <div class="mm-scan-placeholder-desc">
                        Position your face in the center of the frame<br>
                        and click the button below to begin.
                    </div>
                </div>
                """,
                unsafe_allow_html=True,
            )
            _, center, _ = st.columns([1, 1, 1])
            with center:
                with st.container():
                    st.markdown('<div class="mm-start-camera-marker"></div>', unsafe_allow_html=True)
                    if st.button("📷  Start Camera", key="face_start_camera_btn", use_container_width=True):
                        st.session_state.face_scan_camera_started = True
                        st.rerun()
        else:
            camera_photo = st.camera_input("Initiate Biometric Camera Scan", label_visibility="collapsed")
            if camera_photo:
                _analyze_and_display(user, camera_photo.read())

        st.markdown(
            """
            <div class="mm-privacy-note">
                🛡️ <b>Your privacy is our priority.</b><br>
                <span>Images are processed securely and not stored.</span>
            </div>
            """,
            unsafe_allow_html=True,
        )


def _render_upload_tab(user: dict) -> None:
    st.markdown('<p class="mm-scan-label">Upload a Photo for Scanning</p>', unsafe_allow_html=True)
    with st.container(border=True):
        st.markdown('<div class="mm-scan-card-marker"></div>', unsafe_allow_html=True)
        uploaded_image = st.file_uploader(
            "Upload Image for Scanning", type=["jpg", "jpeg", "png"], label_visibility="collapsed",
        )
        if uploaded_image:
            _analyze_and_display(user, uploaded_image.read())

        st.markdown(
            """
            <div class="mm-privacy-note">
                🛡️ <b>Your privacy is our priority.</b><br>
                <span>Images are processed securely and not stored.</span>
            </div>
            """,
            unsafe_allow_html=True,
        )


def _render_tips_footer() -> None:
    tips_html = _img_or_emoji_html(
        ("assets", "illustrations", "tips-plant-books.png"), "🌱📚", "mm-tips-illustration", 130
    )
    st.markdown(
        f"""
        <div class="mm-tips-footer">
            <div>
                <div class="mm-tips-title">💡 Tips for best results</div>
                <div class="mm-tips-desc">Ensure good lighting, face the camera directly and remove any obstructions.</div>
            </div>
            {tips_html}
        </div>
        """,
        unsafe_allow_html=True,
    )


def _load_dashboard_css_if_needed() -> bool:
    """Load dashboard.css directly, independent of render_sidebar().

    Streamlit doesn't mind duplicate <style> tags (they just apply harmlessly),
    so this is a safe redundancy in case this section ever gets rendered
    without render_sidebar() having run first in the same script pass.
    """
    css_path = _resolve_first(
        ("styles", "dashboard.css"),
        ("assets", "styles", "dashboard.css"),
        ("assets", "css", "dashboard.css"),
    )
    if css_path.exists():
        st.markdown(f"<style>{css_path.read_text(encoding='utf-8')}</style>", unsafe_allow_html=True)
        return True
    return False


def render_face_detection_section(user: dict, debug: bool = True) -> None:
    """Render the Face Detection section. The scan/analysis logic is
    identical to the original inline `elif section == "Face Detection":`
    block -- only the layout/markup and the camera-reveal gating are new.
    """
    css_ok = _load_dashboard_css_if_needed()

    if debug and not css_ok:
        css_path = _resolve_first(
            ("styles", "dashboard.css"),
            ("assets", "styles", "dashboard.css"),
            ("assets", "css", "dashboard.css"),
        )
        st.warning(
            "dashboard.css not found -- Face Detection page will look "
            f"unstyled. Checked: `{css_path}` (and sibling candidate roots)."
        )

    _render_page_header()

    tab1, tab2 = st.tabs(["📸  Camera Scanner", "📂  Upload Photo"])
    with tab1:
        _render_camera_tab(user)
    with tab2:
        _render_upload_tab(user)

    _render_tips_footer()


In [ ]:
%%writefile components/relax.py
"""
components/relax.py

Premium "Relax & Recharge" section for MoodMentor.

IMPORTANT: UI ONLY for the core logic. The breathing circle markup and the
mood-based Spotify playlist selection are functionally identical to the
original inline "Relax" branch in app.py -- same three playlist IDs, same
mood-to-playlist mapping, same components.iframe() call.

ONE REAL ADDITION (not fabricated, agreed with you): the playlists that
were previously only auto-selected by mood are now also directly
selectable via "Play Music" (calming) / "Explore Sounds" (lo-fi, reused
for "Nature Sounds") buttons. If the user hasn't clicked either, behavior
is 100% identical to the original -- auto-picks by last mood, same
messaging. Nothing about the original behavior was removed or changed,
only added to.

NOT included, by final decision: "Mood Boost Activities" and
"Recommended for You" sections were built, previewed, then intentionally
removed -- they had no backend behind them and would have looked
clickable without doing anything, which erodes trust. Better to ship
only what's real.

Usage in app.py (replaces the original `elif section == "Relax": ...` body):

    from components.relax import render_relax_section
    ...
    elif section == "Relax":
        render_relax_section(user)
"""

import base64
from datetime import datetime
from pathlib import Path

import streamlit as st
import streamlit.components.v1 as components

from db import get_user_mood_history

_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _img_or_emoji_html(rel_path: tuple, emoji_fallback: str, css_class: str, width_px: int) -> str:
    path = _resolve_first(rel_path)
    b64 = _get_base64_of_file(str(path))
    if b64:
        return (
            f'<img src="data:image/png;base64,{b64}" class="{css_class}" '
            f'width="{width_px}" style="width:{width_px}px;height:auto;" alt="" />'
        )
    return f'<div class="{css_class} {css_class}--emoji">{emoji_fallback}</div>'


# The exact three playlist IDs from your original code -- unchanged.
_PLAYLISTS = {
    "calm":   "https://open.spotify.com/embed/playlist/37i9dQZF1DWZqd5JICZI0u?utm_source=generator",
    "upbeat": "https://open.spotify.com/embed/playlist/37i9dQZF1DXcBWIGoYBM5M?utm_source=generator",
    "lofi":   "https://open.spotify.com/embed/playlist/37i9dQZF1DWWQRwui0ExPn?utm_source=generator",
}


def _render_page_header() -> None:
    now = datetime.now()
    st.markdown(
        f"""
        <div class="mm-page-header">
            <div>
                <h1 class="mm-page-title">🌿 Relax &amp; Recharge</h1>
                <p class="mm-page-subtitle">Take a deep breath, relax your mind, and reset your mood.</p>
            </div>
            <div class="mm-home-datetime">
                📅 {now.strftime('%A, %-d %B %Y')} &nbsp;|&nbsp; 🕐 {now.strftime('%I:%M %p')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_top_cards() -> None:
    """Guided Breathing anchors to the real breathing section below.
    Calming/Upbeat/Nature Sounds Music buttons set a session_state override
    read by _render_music_section() -- Nature Sounds reuses the existing
    lo-fi playlist (closest honest match; no new playlist link invented)."""
    st.markdown(
        """
        <div class="mm-relax-top-row">
            <div class="mm-relax-card mm-relax-card--purple">
                <div class="mm-relax-card-title">Guided Breathing</div>
                <div class="mm-relax-card-desc">Calm your mind with soothing breathing exercises.</div>
                <a href="#mm-breathing-section" class="mm-relax-card-btn">🌬️ Start Breathing</a>
            </div>
            <div class="mm-relax-card mm-relax-card--blue">
                <div class="mm-relax-card-title">Calming Music</div>
                <div class="mm-relax-card-desc">Listen to relaxing sounds and healing melodies.</div>
            </div>
            <div class="mm-relax-card mm-relax-card--green">
                <div class="mm-relax-card-title">Nature Sounds</div>
                <div class="mm-relax-card-desc">Immerse yourself in the sounds of nature.</div>
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    _, c2, c3 = st.columns([1, 1, 1])
    with c2:
        with st.container():
            st.markdown('<div class="mm-relax-btn-marker mm-relax-btn-marker--blue"></div>', unsafe_allow_html=True)
            if st.button("🎵  Play Music", key="relax_play_calm_btn", use_container_width=True):
                st.session_state.relax_playlist_override = "calm"
                st.rerun()
    with c3:
        with st.container():
            st.markdown('<div class="mm-relax-btn-marker mm-relax-btn-marker--green"></div>', unsafe_allow_html=True)
            if st.button("🌿  Explore Sounds", key="relax_play_nature_btn", use_container_width=True):
                st.session_state.relax_playlist_override = "lofi"
                st.rerun()


def _render_music_section(user: dict) -> None:
    st.markdown('<div class="mm-section-heading-row" id="mm-music-section">'
                '<span class="mm-section-heading-icon">🎵</span>'
                '<span class="mm-section-heading-text">Therapy Recommendations</span></div>',
                unsafe_allow_html=True)

    override = st.session_state.get("relax_playlist_override")

    if override:
        # User explicitly picked a playlist -- honor that choice.
        playlist_key = override
        label = "Calming acoustic playlist" if override == "calm" else "Upbeat playlist"
        st.info(f"**Playing:** {label} (your choice). "
                f"[Reset to auto-recommendation](#mm-music-section)")
        if st.button("↺  Back to mood-based recommendation", key="relax_reset_override_btn"):
            st.session_state.relax_playlist_override = None
            st.rerun()
    else:
        # EXACT original logic: auto-pick by last mood entry.
        history = get_user_mood_history(user["id"], limit=1)
        last_mood = history[0]["sentiment"] if history else "Normal"

        if last_mood in ["Sad", "Angry", "Stress", "Fear"]:
            st.warning(f"**Mentor's Advice:** Since your last entry showed you were feeling "
                       f"**{last_mood}**, I recommend 5 minutes of mindful breathing before your "
                       f"next meeting. Listen to this calming acoustic playlist to help you center yourself.")
            playlist_key = "calm"
        elif last_mood in ["Happy", "Amazing"]:
            st.success(f"**Mentor's Advice:** Since your last entry showed you were feeling "
                       f"**{last_mood}**, keep that incredible momentum going! This upbeat playlist "
                       f"is perfect while you work.")
            playlist_key = "upbeat"
        else:
            st.info(f"**Mentor's Advice:** You've been feeling **{last_mood}**. To help you find "
                    f"your flow and stay centered today, try this Lo-Fi Beats playlist.")
            playlist_key = "lofi"

    with st.container(border=True):
        st.markdown('<div class="mm-music-card-marker"></div>', unsafe_allow_html=True)
        components.iframe(_PLAYLISTS[playlist_key], width=300, height=352, scrolling=False)


def _render_breathing_section() -> None:
    st.markdown('<div id="mm-breathing-section"></div>', unsafe_allow_html=True)
    with st.container(border=True):
        st.markdown('<div class="mm-breathing-card-marker"></div>', unsafe_allow_html=True)
        breath_img_html = _img_or_emoji_html(
            ("assets", "illustrations", "breathing-woman.png"), "🧘‍♀️", "mm-breathing-illustration", 130
        )
        st.markdown(
            f"""
            <div class="mm-breathing-row">
                <div>
                    <div class="mm-card-title">🌬️ Guided Breathing</div>
                    <div class="mm-card-subtitle">
                        Follow the circle. Breathe in as it expands, hold, and breathe out as it shrinks.
                    </div>
                    <div class="breathing-container">
                        <div class="circle">Breathe</div>
                    </div>
                </div>
                {breath_img_html}
            </div>
            """,
            unsafe_allow_html=True,
        )
        st.info("💡 **Mentor's Tip:** This 4-7-8 breathing technique activates your parasympathetic "
                "nervous system, reducing anxiety in just 60 seconds.")


def _render_footer_quote() -> None:
    cup_html = _img_or_emoji_html(
        ("assets", "illustrations", "relax-quote-cup.png"), "☕", "mm-relax-quote-illustration", 90
    )
    st.markdown(
        f"""
        <div class="mm-relax-quote-card">
            <div class="mm-relax-quote-text">
                "You deserve this moment. You matter. Take care of your mind,
                just like you take care of everything else."
            </div>
            {cup_html}
        </div>
        """,
        unsafe_allow_html=True,
    )


def render_relax_section(user: dict, debug: bool = True) -> None:
    """Render the Relax section. Breathing markup and Spotify playlist
    selection logic are functionally identical to the original inline
    `elif section == "Relax":` block -- only layout/markup changed, plus
    one disclosed additive feature (manual playlist choice)."""

    if "relax_playlist_override" not in st.session_state:
        st.session_state.relax_playlist_override = None

    if debug:
        css_path = _resolve_first(
            ("styles", "dashboard.css"),
            ("assets", "styles", "dashboard.css"),
            ("assets", "css", "dashboard.css"),
        )
        if css_path.exists():
            st.markdown(f"<style>{css_path.read_text(encoding='utf-8')}</style>", unsafe_allow_html=True)
        else:
            st.warning(
                "dashboard.css not found -- Relax page will look unstyled. "
                f"Checked: `{css_path}` (and sibling candidate roots)."
            )

    _render_page_header()
    _render_top_cards()

    left, right = st.columns([1, 1], gap="large")
    with left:
        _render_breathing_section()
    with right:
        _render_music_section(user)

    _render_footer_quote()

Overwriting components/relax.py


In [ ]:
%%writefile components/dashboard.py
"""
components/dashboard.py

Premium "Dashboard" section for MoodMentor.

IMPORTANT: UI ONLY for data. get_user_mood_history() is the exact same
call as the original inline "Dashboard" branch, and every number shown is
computed from real data -- no fabricated stats.

DISCLOSED SUBSTITUTIONS (read before wiring this up):
    - The reference had "Wellness Chats" and "Relax Sessions" metric
      cards, but nothing in your code counts either of those. Swapped for
      "Total Check-ins" and "Current Streak" instead -- both real,
      computed the same way Home's streak metric already is.
    - Mood distribution / trend / emotions charts are rebuilt as custom
      SVG/CSS visuals instead of st.pyplot/st.line_chart/st.bar_chart, but
      every number in them comes from the identical counting logic as the
      original code. Only the rendering changed.
    - "View all activity" is a disclosed decorative label -- there's no
      separate activity page in your app.
    - The top date badge is informational (shows the actual span of
      loaded history), not an interactive multi-week filter -- that
      filtering doesn't exist in your original code either.
    - Weekly insights text is computed from real counts (dominant mood,
      trend direction), not a generic made-up sentence.
    - Export PDF uses your exact build_pdf_report() and
      get_period_recommendation() -- passed in as a parameter to avoid
      duplicating that (fairly long) function or creating a circular
      import with app.py.

Usage in app.py (replaces the original `elif section == "Dashboard": ...` body):

    from components.dashboard import render_dashboard_section
    ...
    elif section == "Dashboard":
        render_dashboard_section(user, build_pdf_report)
"""

import base64
from datetime import date
from pathlib import Path

import streamlit as st

from db import MOOD_EMOJI, get_user_mood_history
from recommendations import get_period_recommendation

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#3b82f6"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#22c55e"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#f59e0b"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}
MOOD_TO_NUM = {"Happy": 2, "Neutral": 0, "Sad": -1, "Stress": -1, "Angry": -2, "Fear": -2}


def _style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})


_THIS_FILE = Path(__file__).resolve()
_CANDIDATE_ROOTS = [_THIS_FILE.parent.parent, Path.cwd(), Path.cwd().parent]


def _resolve_first(*candidate_relative_paths: tuple) -> Path:
    for root in _CANDIDATE_ROOTS:
        for parts in candidate_relative_paths:
            candidate = root.joinpath(*parts)
            if candidate.exists():
                return candidate
    return _CANDIDATE_ROOTS[0].joinpath(*candidate_relative_paths[0])


@st.cache_data(show_spinner=False)
def _get_base64_of_file(file_path: str) -> str:
    try:
        with open(file_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except FileNotFoundError:
        return ""


def _img_or_emoji_html(rel_path: tuple, emoji_fallback: str, css_class: str, width_px: int) -> str:
    path = _resolve_first(rel_path)
    b64 = _get_base64_of_file(str(path))
    if b64:
        return (
            f'<img src="data:image/png;base64,{b64}" class="{css_class}" '
            f'width="{width_px}" style="width:{width_px}px;height:auto;" alt="" />'
        )
    return f'<div class="{css_class} {css_class}--emoji">{emoji_fallback}</div>'


def _metric_card(icon: str, label: str, value: str, sub_html: str, accent: str) -> str:
    return f"""
    <div class="mm-dash-metric" style="--accent:{accent}">
        <div class="mm-dash-metric-icon">{icon}</div>
        <div>
            <div class="mm-dash-metric-label">{label}</div>
            <div class="mm-dash-metric-value">{value}</div>
            <div class="mm-dash-metric-sub">{sub_html}</div>
        </div>
    </div>
    """


def _render_header(oldest_date, newest_date) -> None:
    st.markdown(
        f"""
        <div class="mm-page-header">
            <div>
                <h1 class="mm-page-title">👋 Welcome back!</h1>
                <p class="mm-page-subtitle">Here's your emotional wellness overview.</p>
            </div>
            <div class="mm-home-datetime">
                📅 {oldest_date.strftime('%b %-d')} &ndash; {newest_date.strftime('%b %-d, %Y')}
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_donut(counts: dict) -> None:
    total = sum(counts.values())
    if total == 0:
        st.caption("No mood data yet.")
        return

    ordered = [(label, count) for label, count in counts.items() if count > 0]
    ordered.sort(key=lambda x: -x[1])

    stops = []
    running_pct = 0.0
    for label, count in ordered:
        pct = 100 * count / total
        color = _style_for(label)["color"]
        stops.append(f"{color} {running_pct:.1f}% {running_pct + pct:.1f}%")
        running_pct += pct
    gradient = ", ".join(stops)

    legend_html = "".join(
        f'<div class="mm-donut-legend-item">'
        f'<span class="mm-donut-legend-dot" style="background:{_style_for(l)["color"]}"></span>'
        f'{l} <b>{100 * c / total:.0f}%</b></div>'
        for l, c in ordered
    )

    st.markdown(
        f"""
        <div class="mm-donut-row">
            <div class="mm-donut-ring" style="background: conic-gradient({gradient});"></div>
            <div class="mm-donut-legend">{legend_html}</div>
        </div>
        """,
        unsafe_allow_html=True,
    )


def _render_trend_line(trend: dict) -> None:
    if not trend:
        st.caption("Not enough data yet for a trend line.")
        return

    dates = list(trend.keys())
    values = list(trend.values())
    vmin, vmax = min(values + [0]), max(values + [0])
    span = (vmax - vmin) or 1

    w, h, pad = 400, 140, 10
    n = len(values)
    step = (w - 2 * pad) / max(n - 1, 1)
    points = []
    for i, v in enumerate(values):
        x = pad + i * step
        y = h - pad - ((v - vmin) / span) * (h - 2 * pad)
        points.append(f"{x:.1f},{y:.1f}")
    polyline = " ".join(points)
    dots = "".join(
        f'<circle cx="{p.split(",")[0]}" cy="{p.split(",")[1]}" r="4" fill="#7c6fe8" />'
        for p in points
    )
    labels_html = "".join(f'<span>{d[5:]}</span>' for d in dates)

    st.markdown(
        f"""
        <svg viewBox="0 0 {w} {h}" class="mm-trend-svg" preserveAspectRatio="none">
            <polyline points="{polyline}" fill="none" stroke="#7c6fe8" stroke-width="2.5"
                      stroke-linecap="round" stroke-linejoin="round"/>
            {dots}
        </svg>
        <div class="mm-trend-labels">{labels_html}</div>
        """,
        unsafe_allow_html=True,
    )


def _render_emotion_bars(emo_counts: dict) -> None:
    if not emo_counts:
        st.caption("No journal-based emotion data yet.")
        return
    max_v = max(emo_counts.values())
    # NOTE: built as single-line HTML per bar (no leading whitespace on any
    # line) -- Markdown treats lines indented 4+ spaces as a literal code
    # block, which was causing this HTML to render as raw escaped text
    # instead of being parsed as HTML.
    bars = []
    for k, v in sorted(emo_counts.items(), key=lambda x: -x[1]):
        pct = 100 * v / max_v
        bars.append(
            f'<div class="mm-bar-col"><div class="mm-bar-track">'
            f'<div class="mm-bar-fill" style="height:{pct:.0f}%"></div></div>'
            f'<div class="mm-bar-label">{k}</div>'
            f'<div class="mm-bar-value">{v}</div></div>'
        )
    bars_html = "".join(bars)
    st.markdown(f'<div class="mm-bar-chart">{bars_html}</div>', unsafe_allow_html=True)


def _render_activity_table(history: list) -> None:
    rows_html = "".join(
        f"""
        <tr>
            <td>{h['mood_date']}</td>
            <td>{h['created_at'].strftime('%H:%M')}</td>
            <td>{_style_for(h['sentiment'])['emoji']} {h['sentiment']}</td>
            <td>{f"{h['confidence']:.0%}" if h.get('confidence') is not None else '—'}</td>
            <td>{h['source']}</td>
        </tr>
        """
        for h in history[:15]
    )
    st.markdown(
        f"""
        <table class="mm-activity-table">
            <thead>
                <tr><th>Date</th><th>Time</th><th>Mood</th><th>Confidence</th><th>Source</th></tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
        <div class="mm-view-all-label">View all activity →</div>
        """,
        unsafe_allow_html=True,
    )


def _compute_weekly_insight(counts: dict, trend: dict) -> str:
    """Real computed insight -- dominant mood + trend direction, not a
    generic made-up sentence."""
    total = sum(counts.values())
    if total == 0:
        return "Log a few moods to start seeing personalized insights here."

    dominant = max(counts.items(), key=lambda x: x[1])[0]
    values = list(trend.values())
    direction = ""
    if len(values) >= 2:
        direction = " up" if values[-1] > values[0] else (" down" if values[-1] < values[0] else " steady")

    return (
        f"Your most common mood recently has been **{dominant}**. "
        f"Your overall trend looks{direction} over this period."
    )


def render_dashboard_section(user: dict, build_pdf_report_fn, debug: bool = True) -> None:
    """Render the Dashboard section. get_user_mood_history() call and all
    counting logic is identical to the original inline
    `elif section == "Dashboard":` block -- only rendering changed.
    build_pdf_report_fn is passed in (your existing build_pdf_report from
    app.py) to avoid duplicating that function or creating a circular
    import.
    """
    if debug:
        css_path = _resolve_first(
            ("styles", "dashboard.css"), ("assets", "styles", "dashboard.css"), ("assets", "css", "dashboard.css"),
        )
        if css_path.exists():
            st.markdown(f"<style>{css_path.read_text(encoding='utf-8')}</style>", unsafe_allow_html=True)
        else:
            st.warning(f"dashboard.css not found. Checked: `{css_path}`")

    history = get_user_mood_history(user["id"], limit=200)
    if not history:
        st.info("No entries yet — pick a mood on Home or write a journal entry to see your dashboard.")
        return

    counts = {}
    for h in history:
        counts[h["sentiment"]] = counts.get(h["sentiment"], 0) + 1

    by_date = {}
    for h in history:
        by_date.setdefault(h["mood_date"], []).append(MOOD_TO_NUM.get(h["sentiment"], 0))
    trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}

    emo_counts = {}
    for h in history:
        if h["source"] == "nlp" and h["emotion"]:
            emo_counts[h["emotion"]] = emo_counts.get(h["emotion"], 0) + 1

    latest = history[0]
    journal_count = sum(1 for h in history if h.get("journal_text"))
    streak = 0
    day_ptr = date.today()
    day_set = {h["mood_date"] for h in history}
    while day_ptr in day_set:
        streak += 1
        day_ptr = date.fromordinal(day_ptr.toordinal() - 1)

    _render_header(history[-1]["mood_date"], history[0]["mood_date"])

    m1, m2, m3, m4 = st.columns(4)
    with m1:
        s = _style_for(latest["sentiment"])
        st.markdown(_metric_card("😊", "Overall Mood", latest["sentiment"], "You've had a good week!", s["color"]), unsafe_allow_html=True)
    with m2:
        st.markdown(_metric_card("📝", "Journal Entries", str(journal_count), "Keep journaling!", "#22c55e"), unsafe_allow_html=True)
    with m3:
        st.markdown(_metric_card("📊", "Total Check-ins", str(len(history)), "All-time entries", "#3b82f6"), unsafe_allow_html=True)
    with m4:
        st.markdown(_metric_card("🔥", "Current Streak", f"{streak} Days", "Keep it going!", "#f59e0b"), unsafe_allow_html=True)

    st.markdown('<div class="mm-card-spacer"></div>', unsafe_allow_html=True)

    c1, c2, c3 = st.columns(3)
    with c1:
        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker"></div>', unsafe_allow_html=True)
            st.markdown('<div class="mm-card-title">Mood distribution</div>', unsafe_allow_html=True)
            _render_donut(counts)
    with c2:
        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker"></div>', unsafe_allow_html=True)
            st.markdown('<div class="mm-card-title">Mood trend over time</div>', unsafe_allow_html=True)
            _render_trend_line(trend)
    with c3:
        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker"></div>', unsafe_allow_html=True)
            st.markdown('<div class="mm-card-title">Emotions from journal entries</div>', unsafe_allow_html=True)
            _render_emotion_bars(emo_counts)

    st.markdown('<div class="mm-card-spacer"></div>', unsafe_allow_html=True)

    left, right = st.columns([1.6, 1], gap="large")
    with left:
        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker"></div>', unsafe_allow_html=True)
            st.markdown('<div class="mm-card-title">Recent activity</div>', unsafe_allow_html=True)
            _render_activity_table(history)

    with right:
        insight_img = _img_or_emoji_html(
            ("assets", "illustrations", "breathing-woman.png"), "🧘‍♀️", "mm-insight-illustration", 90
        )
        insight_text = _compute_weekly_insight(counts, trend)
        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker"></div>', unsafe_allow_html=True)
            st.markdown(
                f"""
                <div class="mm-insight-row">
                    <div>
                        <div class="mm-card-title">✨ Weekly insights</div>
                        <div class="mm-insight-text">{insight_text}</div>
                    </div>
                    {insight_img}
                </div>
                """,
                unsafe_allow_html=True,
            )

        with st.container(border=True):
            st.markdown('<div class="mm-dash-card-marker mm-export-card-marker"></div>', unsafe_allow_html=True)
            st.markdown(
                """
                <div class="mm-card-title">⬇️ Export report</div>
                <div class="mm-card-subtitle">Download your mood and activity report.</div>
                """,
                unsafe_allow_html=True,
            )
            oldest_date = history[-1]["mood_date"]
            today = date.today()
            date_range = st.date_input(
                "Select date range", value=(oldest_date, today),
                min_value=oldest_date, max_value=today,
                key="dashboard_export_range", label_visibility="collapsed",
            )
            with st.container():
                st.markdown('<div class="mm-export-btn-marker"></div>', unsafe_allow_html=True)
                clicked = st.button("Export PDF", key="dashboard_export_btn", use_container_width=True)
            if clicked:
                if isinstance(date_range, tuple) and len(date_range) == 2:
                    start_d, end_d = date_range
                else:
                    start_d = end_d = date_range
                filtered = [h for h in history if start_d <= h["mood_date"] <= end_d]
                if not filtered:
                    st.warning("No entries in that date range.")
                else:
                    recommendation_text = get_period_recommendation(filtered)
                    pdf_bytes = build_pdf_report_fn(
                        user["username"], start_d, end_d, filtered, recommendation_text,
                    )
                    st.success(recommendation_text)
                    st.download_button(
                        "Download PDF", data=pdf_bytes,
                        file_name=f"moodmentor_report_{start_d}_{end_d}.pdf",
                        mime="application/pdf",
                    )

In [ ]:
%%writefile app.py
import os, re, io, calendar
from datetime import date, datetime
import requests, streamlit as st
import matplotlib.pyplot as plt
from components.sidebar import render_sidebar
from components.home import render_home_section
from components.landing import show_landing
from components.wellness_chat import render_wellness_chat_section
from components.face_detection import render_face_detection_section
from components.relax import render_relax_section
from components.auth import render_auth_screen
from components.dashboard import render_dashboard_section
from reportlab.lib.pagesizes import letter
from components.journal import render_journal_section
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from db import (init_db, save_mood_log, save_manual_mood, MOOD_LABELS, MOOD_EMOJI,
                 get_mood_logs_for_month, get_user_mood_history,
                 get_all_employee_mood_logs, get_latest_mood_per_employee, save_face_scan)
from recommendations import get_period_recommendation
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp)
from email_utils import send_otp

st.set_page_config(page_title="MoodMentor", layout="wide")

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

BRAND_GREEN = "#1DBF73"
BRAND_GREEN_DARK = "#159c5e"
INK = "#1f2937"
MUTED = "#6b7280"
BG = "#f5f7f6"

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#2ecc71"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#3498db"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#e67e22"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}
def style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})

MOOD_TO_NUM = {"Happy": 2, "Neutral": 0, "Sad": -1, "Stress": -1, "Angry": -2, "Fear": -2}

def inject_css():
    st.markdown("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&display=swap');

        header {visibility: hidden;}

        /* 1. True Professional Background - Soft Premium Gradient (No messy images) */
        .stApp {
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%) !important;
            background-attachment: fixed !important;
            font-family: 'Outfit', sans-serif !important;
        }

        /* 2. Main Layout Container */
        .block-container {
            padding: 3rem 4rem !important;
            max-width: 1200px !important;
        }

        /* 3. The Sidebar - Deep Professional Slate */
        [data-testid="stSidebar"] {
            background: linear-gradient(180deg, #0f172a 0%, #1e293b 100%) !important;
            border-right: 1px solid rgba(255,255,255,0.05) !important;
            box-shadow: 4px 0 24px rgba(0,0,0,0.1) !important;
        }

        /* Sidebar Text - Crisp White */
        [data-testid="stSidebar"] span, [data-testid="stSidebar"] p, [data-testid="stSidebar"] label, [data-testid="stSidebar"] div {
            color: #f8fafc !important;
            font-weight: 500 !important;
            font-size: 15px !important;
        }

        /* Sidebar Navigation Safe Styling */
        [data-testid="stSidebar"] div[role="radiogroup"] > label {
            background: transparent !important;
            padding: 12px 20px !important;
            border-radius: 12px !important;
            margin-bottom: 8px !important;
            border: none !important;
            box-shadow: none !important;
            transition: all 0.2s ease !important;
        }
        [data-testid="stSidebar"] div[role="radiogroup"] > label:hover {
            background: rgba(255,255,255,0.05) !important;
        }
        [data-testid="stSidebar"] div[role="radiogroup"] > label[data-checked="true"] {
            background: #6366f1 !important; /* Indigo accent */
            box-shadow: 0 4px 15px rgba(99, 102, 241, 0.3) !important;
        }

        /* Remove any weird background from sidebar user info */
        [data-testid="stSidebar"] div[data-testid="stCaptionContainer"] {
            background: transparent !important;
            backdrop-filter: none !important;
            box-shadow: none !important;
        }

        /* 4. Typography Main Area - High Contrast */
        h1 { color: #0f172a !important; font-weight: 700 !important; font-size: 2.5rem !important; letter-spacing: -0.5px !important; }
        h2, h3 { color: #1e293b !important; font-weight: 600 !important; letter-spacing: -0.3px !important; }
        p { color: #475569 !important; font-size: 16px !important; }

        /* 5. Glassmorphism Metric Cards */
        .mm-metric {
            background: rgba(255, 255, 255, 0.7) !important;
            backdrop-filter: blur(10px) !important;
            -webkit-backdrop-filter: blur(10px) !important;
            border-radius: 24px !important;
            padding: 30px !important;
            border: 1px solid rgba(255, 255, 255, 0.8) !important;
            box-shadow: 0 10px 40px rgba(0, 0, 0, 0.03) !important;
            text-align: left !important;
            transition: transform 0.2s ease, box-shadow 0.2s ease !important;
        }
        .mm-metric:hover {
            transform: translateY(-5px) !important;
            box-shadow: 0 15px 50px rgba(0, 0, 0, 0.05) !important;
            background: rgba(255, 255, 255, 0.9) !important;
        }
        .mm-metric .mm-value { color: #0f172a !important; font-size: 38px !important; font-weight: 700 !important; margin-top: 10px !important; }
        .mm-metric .mm-label { color: #6366f1 !important; font-size: 13px !important; font-weight: 700 !important; text-transform: uppercase; letter-spacing: 1px; }

        /* 6. Fix Streamlit Radio Buttons (Emojis) */
        /* We will NOT try to turn them into massive blocks. We will style the container cleanly. */
        .stRadio > div[role="radiogroup"] {
            background: rgba(255, 255, 255, 0.5) !important;
            backdrop-filter: blur(10px) !important;
            padding: 20px !important;
            border-radius: 20px !important;
            border: 1px solid rgba(255, 255, 255, 0.8) !important;
            box-shadow: 0 10px 30px rgba(0,0,0,0.02) !important;
        }
        /* Style individual radio labels safely */
        .stRadio > div[role="radiogroup"] > label {
            background: transparent !important;
            padding: 10px 15px !important;
            border-radius: 10px !important;
            transition: all 0.2s ease !important;
            margin-right: 10px !important;
        }
        .stRadio > div[role="radiogroup"] > label:hover {
            background: rgba(255, 255, 255, 0.8) !important;
        }
        .stRadio > div[role="radiogroup"] > label[data-checked="true"] {
            background: #ffffff !important;
            border: 1px solid #e2e8f0 !important;
            box-shadow: 0 4px 15px rgba(0,0,0,0.05) !important;
        }

        /* 7. Buttons - Premium Solid Indigo */
        .stButton>button, .stFormSubmitButton>button {
            background: #6366f1 !important;
            color: #ffffff !important;
            border-radius: 12px !important;
            padding: 0.6rem 2.5rem !important;
            font-weight: 600 !important;
            border: none !important;
            box-shadow: 0 4px 15px rgba(99, 102, 241, 0.3) !important;
            transition: all 0.2s ease !important;
        }
        .stButton>button:hover, .stFormSubmitButton>button:hover {
            transform: translateY(-2px);
            background: #4f46e5 !important;
            box-shadow: 0 8px 25px rgba(99, 102, 241, 0.4) !important;
        }

        /* Secondary Buttons / Logout */
        [data-testid="stSidebar"] .stButton>button, button[kind="secondary"] {
            background: transparent !important;
            color: #f8fafc !important;
            border: 1px solid rgba(255,255,255,0.2) !important;
            box-shadow: none !important;
        }
        [data-testid="stSidebar"] .stButton>button:hover, button[kind="secondary"]:hover {
            background: rgba(255,255,255,0.1) !important;
            transform: translateY(0) !important;
        }

        /* 8. Container Cards */
        .welcome-box, .auth-card {
            background: rgba(255, 255, 255, 0.7) !important;
            backdrop-filter: blur(10px) !important;
            border-radius: 24px !important;
            padding: 40px !important;
            border: 1px solid rgba(255, 255, 255, 0.8) !important;
            box-shadow: 0 10px 40px rgba(0, 0, 0, 0.03) !important;
        }
    </style>
    """, unsafe_allow_html=True)

def donut_chart(counts: dict, size=2.6):
    labels, values, colors = [], [], []
    for k, v in counts.items():
        if v > 0:
            labels.append(k); values.append(v)
            colors.append(style_for(k)["color"])
    if not values:
        return None
    fig, ax = plt.subplots(figsize=(size, size))
    ax.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor="white"))
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    return fig

def metric_tile(label, value, sub=None):
    sub_html = f"<div class='mm-sub'>{sub}</div>" if sub else ""
    st.markdown(
        f"<div class='mm-metric'><div class='mm-label'>{label}</div>"
        f"<div class='mm-value'>{value}</div>{sub_html}</div>",
        unsafe_allow_html=True,
    )

def build_pdf_report(username, start_d, end_d, entries, recommendation_text):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=letter, topMargin=48, bottomMargin=48)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("MoodMentor Wellness Report", styles["Title"]))
    story.append(Paragraph(f"{username} &nbsp;|&nbsp; {start_d} to {end_d}", styles["Normal"]))
    story.append(Spacer(1, 16))

    counts = {}
    for h in entries:
        counts[h["sentiment"]] = counts.get(h["sentiment"], 0) + 1
    summary_line = ", ".join(f"{k}: {v}" for k, v in counts.items())
    story.append(Paragraph("Mood summary", styles["Heading2"]))
    story.append(Paragraph(f"{len(entries)} entries logged. {summary_line}.", styles["Normal"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("Recommendation", styles["Heading2"]))
    story.append(Paragraph(recommendation_text, styles["Normal"]))
    story.append(Spacer(1, 16))

    story.append(Paragraph("Entries", styles["Heading2"]))
    table_data = [["Date", "Time", "Mood", "Emotion", "Confidence", "Source"]]
    for h in sorted(entries, key=lambda r: r["created_at"], reverse=True):
        table_data.append([
            str(h["mood_date"]),
            h["created_at"].strftime("%H:%M"),
            h["sentiment"] or "\u2014",
            h.get("emotion") or "\u2014",
            f"{h['confidence']:.0%}" if h.get("confidence") is not None else "\u2014",
            h["source"],
        ])
    tbl = Table(table_data, repeatRows=1, hAlign="LEFT")
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1DBF73")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#dddddd")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f7f6")]),
    ]))
    story.append(tbl)

    doc.build(story)
    buf.seek(0)
    return buf.getvalue()


inject_css()

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "welcome"
if "show_auth_panel" not in st.session_state: st.session_state.show_auth_panel = False
if "auth_mode" not in st.session_state: st.session_state.auth_mode = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "cal_year" not in st.session_state: st.session_state.cal_year = date.today().year
if "cal_month" not in st.session_state: st.session_state.cal_month = date.today().month
if "today_mood_saved" not in st.session_state: st.session_state.today_mood_saved = False
if "nav" not in st.session_state: st.session_state.nav = "Home"

def goto_auth(mode): st.session_state.auth_mode = mode; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)


if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        role = user.get("role", "employee")
        headers = {"Authorization": f"Bearer {st.session_state.token}"}

        if role == "employee":
            nav_options = ["Home", "Journal", "Wellness Chat", "Face Detection", "Relax", "Dashboard"]
        else:
            nav_options = ["Reports"]
        render_sidebar(user, role, nav_options)



        if role == "employee":
            section = st.session_state.nav

            if section == "Home":
                render_home_section(user)


            elif section == "Journal":
                render_journal_section(user, st.session_state.token)

            elif section == "Wellness Chat":
                render_wellness_chat_section(st.session_state.token)

            elif section == "Face Detection":
                render_face_detection_section(user)

            elif section == "Relax":
                render_relax_section(user)

            elif section == "Dashboard":
                render_dashboard_section(user, build_pdf_report)

        else:
            st.subheader("Employee Wellness Report")

            latest = get_latest_mood_per_employee()
            if not latest:
                st.info("No employee entries yet.")
            else:
                st.write("**Latest mood per employee**")
                table_rows = [{
                    "Employee": row["username"],
                    "Email": row["email"],
                    "Date": row["mood_date"],
                    "Time": row["created_at"].strftime("%H:%M"),
                    "Mood": f"{style_for(row['sentiment'])['emoji']} {row['sentiment']}",
                    "Emotion": row["emotion"],
                } for row in latest]
                st.dataframe(table_rows, use_container_width=True)
            st.markdown("</div>", unsafe_allow_html=True)

            st.write("**Team mood trend (last 30 days)**")
            history = get_all_employee_mood_logs(limit_days=30)
            if not history:
                st.info("Not enough data yet to draw a trend chart.")
            else:
                by_date = {}
                for row in history:
                    d = row["mood_date"]
                    by_date.setdefault(d, []).append(MOOD_TO_NUM.get(row["sentiment"], 0))
                trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                st.line_chart(trend)
                st.caption("Average mood score per day across all employees "
                           "(2 = Happy, 0 = Neutral, -1 = Sad/Stress, -2 = Angry/Fear)")
            st.markdown("</div>", unsafe_allow_html=True)

        st.stop()
    st.session_state.token = None


if st.session_state.page == "welcome":

    if not st.session_state.show_auth_panel:
        show_landing()
        st.stop()

    render_auth_screen()

    st.stop()



In [ ]:
%%writefile nlp_pipeline.py
"""
nlp_pipeline.py
Multilingual NLP pipeline for employee feedback:
normalize -> detect language -> clean -> tokenize -> stopword-filter ->
translate to English -> lemmatize -> sentiment (VADER) -> emotion (BERT).

Stopword filtering uses the `stopwordsiso` package, which ships stopword
sets for 50+ languages keyed by ISO 639-1 code (the same codes langdetect
returns), so any supported language is handled automatically instead of
needing a hardcoded list per language. If the detected language isn't in
stopwordsiso's coverage, filtering is simply skipped for that text.

Heavy libs (spacy model, translator, vader, BERT emotion model, Qwen chat
model) load once at import time via lazy module-level globals, so repeated
/analyze calls reuse them.
"""

import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from recommendations import get_recommendation, WELLNESS_RECOMMENDATIONS

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}


def _get_stopwords(language_code: str) -> set:
    """
    Returns the stopword set for `language_code` using stopwordsiso, which
    covers 50+ languages by ISO 639-1 code. Returns an empty set for any
    language it doesn't cover -- filtering is skipped rather than failing,
    so unsupported languages still flow through the rest of the pipeline.
    """
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]


GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}




def _get_nlp():
    """Lazy-load the multilingual spaCy model once per process."""
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp


def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader


def _get_qwen():
    """Lazy-load Qwen2.5-0.5B-Instruct once per process (GPU if available).
    Still used by the wellness chatbot (wellness_chat_reply) -- only the
    emotion-detection step now uses BERT instead."""
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer


def _get_bert_emotion_pipeline():
    """
    Lazy-load the fine-tuned BERT emotion classifier once per process, using
    Hugging Face's `pipeline()` helper -- this bundles the tokenizer and the
    model together so we just call it with raw text and get scores back.

    `top_k=None` tells the pipeline to return a score for every label
    instead of just the single top prediction, so we can build a full
    scores dict (matching what the UI already expects).
    """
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline


def _bert_emotion(text: str) -> dict:
    """
    Classifies `text` using the fine-tuned BERT GoEmotions model, then maps
    the 28 GoEmotions labels down to our 6 app-level EMOTION_LABELS by
    summing mapped scores. Returns the same shape the rest of the app
    already expects: {"emotion": <label>, "scores": {label: 0-1, ...}}.
    """
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    confidence = app_scores[final_emotion]
    return {"emotion": final_emotion, "scores": app_scores, "confidence": confidence}


def process_employee_feedback(text: str) -> dict:
    """Runs the full pipeline on a single blob of text and returns a results dict."""
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive"
    elif compound_score <= -0.05:
        final_sentiment = "Negative"
    else:
        final_sentiment = "Neutral"

    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    emotion_confidence = bert_result["confidence"]

    # BERT's "Neutral" bucket sums 5 GoEmotions sub-labels (neutral,
    # realization, surprise, curiosity, desire) vs. 1-4 for the other
    # buckets, so on short/ambiguous text it wins the argmax by default
    # even when VADER clearly reads the text as negative (e.g. "stressed").
    # Previously this mismatch was only patched inside get_recommendation()
    # for the *recommendation text* -- the emotion label shown to the user
    # and saved to mood_logs.emotion stayed "Neutral", contradicting the
    # sentiment shown right next to it and skewing the Dashboard's
    # "Emotions detected" chart toward Neutral. Apply the same override to
    # final_emotion itself so what's displayed/stored/charted is consistent
    # with what's recommended.
    if final_emotion_label == "Neutral" and final_sentiment == "Negative":
        final_emotion_label = "Sad"

    final_emotion = final_emotion_label
    recommendation = get_recommendation(final_emotion_label, emotion_confidence, final_sentiment, compound_score)

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
        "emotion_confidence": emotion_confidence,
        "recommendation": recommendation,
    }


CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)


def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)


def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:
    """
    Generates a supportive wellness chatbot reply using the Qwen chat model.
    (The chatbot still uses Qwen -- it needs to generate free-form
    conversational replies, which is a generation task, not a
    classification task, so BERT isn't a fit here.)

    `history` is an optional list of {"role": "user"|"assistant", "content": str}
    dicts representing prior turns in the conversation (kept short/recent by
    the caller — this function does not trim it).

    Always checks for crisis language first; if found, returns a fixed,
    resource-pointing message instead of an LLM-generated one, since we
    never want a small model improvising in a safety-critical moment.
    """
    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}



In [ ]:
from db import cursor

with cursor(commit=True) as cur:
    # 1. Normalize any case-variant spellings of Neutral to the canonical form.
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Neutral' "
        "WHERE emotion ILIKE 'neutral' AND emotion != 'Neutral'"
    )
    normalized = cur.rowcount

    # 2. Re-apply the Neutral+Negative -> Sad override to rows saved before the fix.
    #    sentiment here is the mapped 5-point label ('Sad'), set by NLP_TO_MOOD_LABEL
    #    from the pipeline's original 'Negative' sentiment -- see db.save_mood_log().
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Sad' "
        "WHERE emotion = 'Neutral' AND sentiment = 'Sad' AND source = 'nlp'"
    )
    relabeled = cur.rowcount

print(f"Normalized casing on {normalized} row(s); relabeled {relabeled} Neutral->Sad row(s).")

In [ ]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

app.add_middleware(CORSMiddleware, allow_origins=["*"],
                    allow_methods=["*"], allow_headers=["*"])

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }


def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    """
    Returns (text_blob, used_column). For TXT, used_column is None.
    For CSV, joins all non-empty values of the chosen column (or the last
    column if none/invalid was specified) into one whitespace-joined blob —
    matches the notebook's "whole file as one blob" behavior.
    """
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]


@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    """
    Runs the multilingual NLP pipeline (language detection, cleaning,
    stopword filtering, translation, lemmatization, VADER sentiment,
    keyword-based emotion) on an uploaded .csv or .txt file.
    """
    get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results



class TextIn(BaseModel):
    text: str

@app.post("/analyze-text")
async def analyze_text(payload: TextIn, authorization: str = Header(None)):
    """Same NLP pipeline as /analyze, but for text typed directly into the
    Journal tab's textbox instead of an uploaded file."""
    get_user(authorization)

    text_blob = payload.text.strip()
    if not text_blob:
        raise HTTPException(400, "Text cannot be empty.")

    results = process_employee_feedback(text_blob)
    results["filename"] = None
    results["file_type"] = "TEXT"
    results["used_column"] = None
    results["original_char_count"] = len(text_blob)
    return results

class ChatTurn(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []


@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    """
    Wellness support chatbot endpoint. Stateless on the server: the client
    (Streamlit) sends the recent conversation history along with each new
    message, and we generate the next reply with the same Qwen model used
    for emotion detection.
    """
    get_user(authorization)

    message = payload.message.strip()
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [turn.dict() for turn in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


In [ ]:
from db import init_db
init_db()
print("Connected to PostgreSQL and ensured tables exist.")

Connected to PostgreSQL and ensured tables exist.


In [ ]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

# Kill any previous tunnels/streamlit/uvicorn instances from earlier runs in this session
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

# Launch FastAPI (backend.py) in the background on port 8000 (internal only, not tunneled)
get_ipython().system_raw(
    'uvicorn backend:app --host 0.0.0.0 --port 8000 &'
)
time.sleep(5)
# Launch Streamlit in the background, quietly, on port 8501
get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection false &'
)
time.sleep(4)  # give both servers a moment to boot

public_url = ngrok.connect(8501, "http")
print(f" Your app is live at: {public_url}")

 Your app is live at: NgrokTunnel: "https://unsaid-exterior-obedient.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.
